# Giga Meter — Fleet Profile

How each country's fleet actually measures: rhythm, seasonality, silence and churn,
computed **server-side in Trino** so it scales across the whole fleet. Only small
aggregates come back — a profile row, twelve seasonal rows and a spell histogram
per country — never the row-level measurements.

**What it answers**
- What is the measurement rhythm — near-daily, weekly, sparse — and how long do schools last?
- When is each country's low season, learned from its own data rather than assumed?
- Given a school has been silent X days, how likely is it to come back? (Kaplan-Meier, censoring-corrected)
- How much churn does each fleet carry, expressed as incidence per 100 school-years so countries are comparable?
- Do devices keep the 48-a-day connection-check beat, how far into the day does each fleet last, when a scheduled speed test goes missing was the machine off or did the test fail, and is the loss the device or the link? (Parts 3f–3j)
- How much measuring actually happens per day, how concentrated is it in a few schools, and how many readings does a school's median need? (Parts 3k–3m)

**Design notes**
- Every metric is **exposure-corrected**: seasonal averages divide by school-months *in tenure*, not by calendar months present, and silence spells are treated as survival data with censoring at the last date in the data.
- Recency is measured against each country's own last data date, so a stale pipeline in one country does not look like churn.
- **Reactivation campaigns are registered explicitly** (see the CAMPAIGNS cell) — Mongolia ran one Mar–May 2026 and Sri Lanka is running one now, so their return rates and churn describe a fleet under intervention.
- The low season comes from the **published school calendar**, confirmed against the measurement dip; a purely data-derived rule produces false positives in countries with one extreme peak month.

---
## Part 0 — Setup

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
from pathlib import Path
from datetime import date, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pytz

from IPython.display import display

# Optional connectivity libs (only needed when USE_CACHED_DATA = False)
try:
    import delta_sharing
    DELTA_SHARING_AVAILABLE = True
except ImportError:
    DELTA_SHARING_AVAILABLE = False
    print("⚠️ delta_sharing not available - will use cached data only")

try:
    import trino
    from trino.dbapi import connect
    TRINO_AVAILABLE = True
except ImportError:
    TRINO_AVAILABLE = False
    print("⚠️ trino not available - will use cached data only")

# -----------------------------------------------------------------------------
# Data-loading helpers (bundled in ./helpers)
#   load_master            - school master via Delta Sharing / Trino, CSV-cached
#   load_measurements      - country measurements via Trino, parquet-cached + incremental
#   format_measurements    - query builder + light post-processing for the
#                            consolidated table default.all_gigameter_measurement_data
#   get_trino_cursor/engine - PRD Trino over kubectl port-forward (auto-started)
# -----------------------------------------------------------------------------
set_up_dir = Path.cwd() / "helpers"
if str(set_up_dir) not in sys.path:
    sys.path.insert(0, str(set_up_dir))

try:
    import format_measurements
    from load_master import load_master, load_master_trino
    from load_measurements import (
        load_measurements,
        load_registration,
        get_trino_cursor,
        get_trino_engine,
    )
    HELPERS_AVAILABLE = True
except ImportError as e:
    HELPERS_AVAILABLE = False
    print(f"⚠️ data-loading helpers not importable from {set_up_dir}: {e}")

# -----------------------------------------------------------------------------
# Analysis helpers + Giga chart style (moved out of the notebook)
#   eda_helpers      - education inference, ISP canonicalisation, IQB-Edu engine,
#                      legacy service-tier scaffolding
#   giga_chart_style - fonts + palette + rcParams (applied on import)
# -----------------------------------------------------------------------------
from eda_helpers import (resolve_country, infer_edlevel_from_name, clean_isp, build_isp_canon,
                         IQB_CONFIG, IQB_USE_CASES, IQB_PERCENTILES, IQB_BENCHMARK,
                         MIN_MEASUREMENTS_FOR_IQB, calculate_iqb_score,
                         _config_for_use_case,
                         classify_service_level, tier_order,
                         TIER_THRESHOLD_1, TIER_THRESHOLD_2, TIER_THRESHOLD_3,
                         paired_shift_test, two_group_shift_test,
                         format_shift_result, bootstrap_ci,
                         wilson_ci, fmt_pct_ci,
                         kruskal_omnibus, pairwise_shift_tests)
from giga_chart_style import (GIGA_PRIMARY, GIGA_GREY, GIGA_BLUE, GIGA_GOOD,
                              GIGA_MODERATE, GIGA_BAD, GIGA_TIER_RAMP, GIGA_CYCLE,
                              GIGA_SUPTITLE)

print("\u2713 Imports complete \u00b7 Giga chart style applied (Open Sans / Manrope, Giga palette)")


In [ ]:
# =============================================================================
# SCOPE — which countries, and where to read from
# =============================================================================
COUNTRIES = []                        # ISO3 codes; [] = every country in the table
SOURCE_FILTER = "GigaMeter"           # rt_source; None = all sources
USE_TRINO = True                      # False = build from local country parquet caches
MIN_SCHOOLS = 20                      # skip countries with fewer measuring schools

CACHE_ROOT = Path("./cache")
LOW_SEASON_FRAC = 0.75                # month is "low season" below this share of the MEDIAN month
MIN_SPAN_MONTHS_FOR_SEASON = 24       # below this, seasonality is provisional (one cycle or less)

EXCLUDE_COUNTRIES = ['KAZ']           # kept out of fleet aggregates (deployment not comparable)

_iso_sql = "" if not COUNTRIES else "AND iso3_code IN ('" + "', '".join(COUNTRIES) + "')"
if EXCLUDE_COUNTRIES:
    _iso_sql += " AND iso3_code NOT IN ('" + "', '".join(EXCLUDE_COUNTRIES) + "')"
_src_sql = "" if not SOURCE_FILTER else f"AND rt_source = '{SOURCE_FILTER}'"
_scope_sql = f"{_iso_sql} {_src_sql}"
print(f"scope: {COUNTRIES or 'ALL countries'} minus {EXCLUDE_COUNTRIES or 'none'} · "
      f"source={SOURCE_FILTER or 'all'} · "
      f"{'Trino' if USE_TRINO else 'local caches'}")

In [ ]:
# =============================================================================
# TIMEZONE INTEGRITY — one corrected local timestamp for the whole notebook
# =============================================================================
# `local_created_timestamp` is not localised for every country: where the country
# is missing from the upstream `default.country_timezones` lookup, the pipeline
# silently falls back to 'UTC' (COALESCE(tz.timezone, tz_name.timezone, 'UTC')),
# so the "local" value is really UTC. Detect it by comparing the offset the data
# implies against the country's real one, then repair from `created_timestamp`,
# which is genuine UTC everywhere. `_LTS` is the expression every query below
# uses in place of the raw column.
TZ_TOL_HOURS = 1                  # slack vs the country's UTC offset (DST is 1 hour)

Q_TZ_AUDIT = f"""
SELECT iso3_code, count(*) AS n,
       approx_percentile(mod(CAST(local_hour_of_measurement AS BIGINT)
                             - hour(created_timestamp) + 24, 24), 0.5) AS obs_off
FROM default.all_gigameter_measurement_data
WHERE local_created_timestamp IS NOT NULL AND created_timestamp IS NOT NULL
  AND local_hour_of_measurement IS NOT NULL
  {_scope_sql}
GROUP BY 1
"""

def _expected_offsets(iso3):
    """Winter and summer UTC offsets, or (None, None) if the zone is unknown."""
    try:
        zone = resolve_country(iso3)['timezone']
    except Exception:
        return None, None
    tz = pytz.timezone(zone)
    return zone, {int(round(tz.utcoffset(datetime(2026, mo, 15)).total_seconds() / 3600)) % 24
                  for mo in (1, 7)}

TZ_FIX, TZ_DROP = {}, []
try:
    from datetime import datetime
    _c = get_trino_cursor()
    _c.execute(Q_TZ_AUDIT)
    _tza = pd.DataFrame(_c.fetchall(), columns=[d[0] for d in _c.description])
    for _r in _tza.itertuples():
        _zone, _exp = _expected_offsets(_r.iso3_code)
        if _exp is None:
            TZ_DROP.append((_r.iso3_code, int(_r.n)))
        elif min(abs(((int(_r.obs_off) - _e + 12) % 24) - 12) for _e in _exp) > TZ_TOL_HOURS:
            TZ_FIX[_r.iso3_code] = _zone
except Exception as _e:
    print(f"⚠️  timezone audit skipped ({_e}) — using local_created_timestamp as shipped")

_LTS = "local_created_timestamp"
if TZ_FIX:
    _cases = "\n           ".join(
        f"WHEN '{k}' THEN CAST(created_timestamp AT TIME ZONE '{v}' AS timestamp)"
        for k, v in sorted(TZ_FIX.items()))
    _LTS = f"(CASE iso3_code\n           {_cases}\n           ELSE local_created_timestamp END)"
_TZ_DROP_SQL = ("AND iso3_code NOT IN ('" + "', '".join(c for c, _ in TZ_DROP) + "')") if TZ_DROP else ""

print(f"timezone repaired (stored 'local' was UTC): "
      f"{', '.join(f'{k} -> {v}' for k, v in sorted(TZ_FIX.items())) or 'none needed'}")
if TZ_DROP:
    print(f"excluded, no timezone on file: "
          f"{', '.join(f'{c} ({n:,} rows)' for c, n in sorted(TZ_DROP, key=lambda x: -x[1]))}")
print("→ every query below keys on the corrected local date via _LTS")

In [ ]:
# =============================================================================
# CAMPAIGN REGISTRY — outreach that moves the numbers, recorded explicitly
# =============================================================================
# Reactivation campaigns manufacture returns. Any country with one inside the
# observation window will show inflated return probabilities and deflated churn
# for that period, and the effect is NOT seasonality — so it has to be recorded
# and controlled for, not absorbed into the baseline.
#
# Add entries as campaigns happen: ISO3 -> list of (start, end or None, label).
CAMPAIGNS = {
    'MNG': [('2026-03-01', '2026-05-31', 'reactivation campaign')],
    'LKA': [('2026-07-01', None,         'reactivation campaign (ongoing)')],
}

def campaign_windows(iso3):
    """[(start_ts, end_ts_or_None, label)] for a country."""
    return [(pd.Timestamp(a), pd.Timestamp(b) if b else None, lbl)
            for a, b, lbl in CAMPAIGNS.get(iso3, [])]

def in_campaign(iso3, when):
    for _a, _b, _ in campaign_windows(iso3):
        if when >= _a and (_b is None or when <= _b):
            return True
    return False

_reg = pd.DataFrame([{'iso3_code': k, 'window': f"{a} → {b or 'ongoing'}", 'label': lbl}
                     for k, v in CAMPAIGNS.items() for a, b, lbl in v])
print("Campaigns on record (edit CAMPAIGNS above as new ones run):")
display(_reg)
print("Countries in this run WITH a campaign in-window: "
      f"{[i for i in COUNTRIES if i in CAMPAIGNS] or 'none'}")

---
## Part 1 — Server-side aggregation

Four small result sets per country. The row-level table is touched once inside the
CTE chain; everything returned is already aggregated.

In [ ]:
# =============================================================================
# SQL — the shared CTE chain (school x day, gaps, tenure) reused by every query
# =============================================================================
# Audited Aug 2026. Choices that matter, stated rather than implied:
#  · the day key is the LOCAL date (`local_created_timestamp`), not the UTC `date`
#    column — they disagree on 4.3% of rows, so every query must use the same one
#  · `ref_d` is each country's own last data date, so a lagging pipeline in one
#    country does not read as churn (verified: no future-dated rows in this table)
#  · SOURCE_FILTER='GigaMeter' excludes the Mlab feed (543k rows, 2,541 schools,
#    all but 39 of which also report via Giga Meter) — school-side app only
#  · seasonality exposure runs first→last measurement, so a school contributes no
#    months after it stops: the seasonal curve describes ACTIVE schools' rhythm
#    and cannot be read as churn
BASE_CTE = f"""
WITH sd AS (                      -- one row per school-day, the only heavy step
    SELECT iso3_code, school_id_giga, CAST({_LTS} AS date) AS d
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_iso_sql} {_src_sql}
    GROUP BY 1, 2, 3
),
ref AS (SELECT iso3_code, max(d) AS ref_d, min(d) AS first_d FROM sd GROUP BY 1),
lagged AS (
    SELECT iso3_code, school_id_giga, d,
           LAG(d) OVER (PARTITION BY iso3_code, school_id_giga ORDER BY d) AS prev_d
    FROM sd
),
gaps AS (
    SELECT iso3_code, school_id_giga, date_diff('day', prev_d, d) AS dur
    FROM lagged WHERE prev_d IS NOT NULL
),
ten AS (
    SELECT iso3_code, school_id_giga, min(d) AS first_d, max(d) AS last_d,
           count(*) AS days_measured,
           date_diff('day', min(d), max(d)) AS tenure_days
    FROM sd GROUP BY 1, 2
)
"""

# 1) one profile row per country
Q_PROFILE = BASE_CTE + """
, per_school AS (
    SELECT t.iso3_code, t.school_id_giga, t.days_measured, t.tenure_days,
           date_diff('day', t.last_d, r.ref_d) AS days_silent,
           t.days_measured / GREATEST(t.tenure_days / 30.44, 1.0) AS days_per_month
    FROM ten t JOIN ref r ON r.iso3_code = t.iso3_code
)
SELECT p.iso3_code,
       count(*) AS schools,
       approx_percentile(p.tenure_days, 0.5) / 30.44 AS median_tenure_months,
       approx_percentile(p.days_per_month, 0.5) AS median_days_per_month,
       approx_percentile(p.days_per_month, 0.25) AS p25_days_per_month,
       approx_percentile(p.days_per_month, 0.75) AS p75_days_per_month,
       sum(p.days_measured) AS school_days,
       sum(date_diff('day', t2.first_d, r.ref_d)) / 365.25 AS school_years_observed,
       -- exposure in which a silence of N days COULD have been observed: school-time
       -- beginning at least N days before the last data date (one column per horizon)
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) -   7, 0)) / 365.25 AS school_years_at_risk_7,
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) -  14, 0)) / 365.25 AS school_years_at_risk_14,
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) -  30, 0)) / 365.25 AS school_years_at_risk_30,
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) -  90, 0)) / 365.25 AS school_years_at_risk_90,
       sum(GREATEST(date_diff('day', t2.first_d, r.ref_d) - 182, 0)) / 365.25 AS school_years_at_risk_182,
       -- % silent must divide by schools that COULD be seen silent that long:
       -- a school first measuring 40 days ago cannot be 90 days silent, and
       -- counting it in the denominator flatters young fleets
       count_if(date_diff('day', t2.first_d, r.ref_d) >= 30)  AS n_eligible_30d,
       count_if(date_diff('day', t2.first_d, r.ref_d) >= 90)  AS n_eligible_90d,
       count_if(date_diff('day', t2.first_d, r.ref_d) >= 182) AS n_eligible_182d,
       100.0 * count_if(p.days_silent > 30)
             / NULLIF(count_if(date_diff('day', t2.first_d, r.ref_d) >= 30), 0)  AS pct_silent_30d,
       100.0 * count_if(p.days_silent > 90)
             / NULLIF(count_if(date_diff('day', t2.first_d, r.ref_d) >= 90), 0)  AS pct_silent_90d,
       100.0 * count_if(p.days_silent > 182)
             / NULLIF(count_if(date_diff('day', t2.first_d, r.ref_d) >= 182), 0) AS pct_silent_182d,
       min(r.first_d) AS data_from, min(r.ref_d) AS data_to
FROM per_school p
JOIN ten t2 ON t2.iso3_code = p.iso3_code AND t2.school_id_giga = p.school_id_giga
JOIN ref r  ON r.iso3_code = p.iso3_code
GROUP BY p.iso3_code
"""

# 2) gap-shape + weekday rhythm per country
Q_RHYTHM = BASE_CTE + """
-- day_of_week(): 1 = Monday ... 7 = Sunday, so <= 5 is Mon-Fri
, wk AS (
    SELECT iso3_code, 100.0 * count_if(day_of_week(d) <= 5) / count(*) AS pct_weekday
    FROM sd GROUP BY 1
)
SELECT g.iso3_code,
       100.0 * count_if(g.dur = 1) / count(*)              AS pct_gap_1d,
       100.0 * count_if(g.dur BETWEEN 2 AND 7) / count(*)  AS pct_gap_2_7d,
       100.0 * count_if(g.dur BETWEEN 8 AND 30) / count(*) AS pct_gap_8_30d,
       100.0 * count_if(g.dur > 30) / count(*)             AS pct_gap_over_30d,
       max(w.pct_weekday)                                  AS pct_weekday
FROM gaps g JOIN wk w ON w.iso3_code = g.iso3_code
GROUP BY g.iso3_code
"""

# 3) silence-spell histogram (completed + censored) — the input to Kaplan-Meier
Q_SPELLS = BASE_CTE + """
SELECT iso3_code, dur, 1 AS returned, count(*) AS n
FROM gaps WHERE dur >= 2 GROUP BY 1, 2
UNION ALL
SELECT t.iso3_code, date_diff('day', t.last_d, r.ref_d) AS dur, 0 AS returned, count(*) AS n
FROM ten t JOIN ref r ON r.iso3_code = t.iso3_code
WHERE date_diff('day', t.last_d, r.ref_d) >= 2
GROUP BY 1, 2
"""

# 4) exposure-corrected seasonality: mean measuring days per school-month IN TENURE
Q_SEASON = BASE_CTE + """
, spine AS (
    SELECT t.iso3_code, t.school_id_giga, m
    FROM ten t
    CROSS JOIN UNNEST(sequence(date_trunc('month', t.first_d),
                               date_trunc('month', t.last_d),
                               INTERVAL '1' MONTH)) AS x(m)
),
per_month AS (
    SELECT iso3_code, school_id_giga, date_trunc('month', d) AS m, count(*) AS days
    FROM sd GROUP BY 1, 2, 3
)
SELECT s.iso3_code, month(s.m) AS calendar_month,
       avg(COALESCE(p.days, 0)) AS mean_days_per_school_month,
       count(*) AS school_months
FROM spine s
LEFT JOIN per_month p
       ON p.iso3_code = s.iso3_code AND p.school_id_giga = s.school_id_giga AND p.m = s.m
GROUP BY 1, 2
"""
# 5) SCHOOL-level view: each school's FIRST pause of >= 2 days, so a habitual
#    pauser counts once rather than dozens of times. Same (dur, returned) shape
#    as Q_SPELLS, which lets the two be compared side by side.
Q_FIRST_SPELL = BASE_CTE + """
, spells AS (
    SELECT iso3_code, school_id_giga, d AS spell_end, date_diff('day', prev_d, d) AS dur, 1 AS returned
    FROM lagged WHERE prev_d IS NOT NULL AND date_diff('day', prev_d, d) >= 2
    UNION ALL
    SELECT t.iso3_code, t.school_id_giga, NULL, date_diff('day', t.last_d, r.ref_d), 0
    FROM ten t JOIN ref r ON r.iso3_code = t.iso3_code
    WHERE date_diff('day', t.last_d, r.ref_d) >= 2
),
ranked AS (
    SELECT iso3_code, school_id_giga, dur, returned,
           ROW_NUMBER() OVER (PARTITION BY iso3_code, school_id_giga
                              ORDER BY COALESCE(spell_end, DATE '9999-12-31')) AS rn
    FROM spells
)
SELECT iso3_code, dur, returned, count(*) AS n
FROM ranked WHERE rn = 1 GROUP BY 1, 2, 3
"""

print("5 queries built · the row-level table is scanned once per query, results are aggregates")

In [ ]:
# =============================================================================
# RUN — Trino if available, else rebuild the same aggregates from local caches
# =============================================================================
def _from_trino():
    cur = get_trino_cursor()
    if cur is None:
        raise RuntimeError("no Trino cursor")
    def q(sql):
        cur.execute(sql)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
    return q(Q_PROFILE), q(Q_RHYTHM), q(Q_SPELLS), q(Q_SEASON), q(Q_FIRST_SPELL)

def _from_cache():
    """Same aggregates from <Country>/*_measurements.parquet — for offline work."""
    prof, rhy, spl, sea = [], [], [], []
    for _dir in sorted(CACHE_ROOT.iterdir()):
        if not _dir.is_dir():
            continue
        _f = sorted(_dir.glob('*_measurements.parquet'))
        if not _f:
            continue
        _iso = _dir.name[:3].upper()
        if COUNTRIES and not any(_dir.name.lower().startswith(c.lower()[:3]) or _iso == c
                                 for c in COUNTRIES):
            _match = [c for c in COUNTRIES if c in COUNTRY_ALIASES.get(_dir.name, [_iso])]
            if not _match:
                continue
            _iso = _match[0]
        _m = pd.read_parquet(_f[0], columns=['school_id_giga', 'date'])
        _m['d'] = pd.to_datetime(_m['date'], errors='coerce', utc=True).dt.tz_localize(None).dt.normalize()
        _sd = _m.dropna(subset=['d']).drop_duplicates(['school_id_giga', 'd']).sort_values(['school_id_giga', 'd'])
        _ref = _sd['d'].max()
        _sd['gap'] = _sd.groupby('school_id_giga')['d'].diff().dt.days
        _t = _sd.groupby('school_id_giga')['d'].agg(first_d='min', last_d='max', days_measured='size')
        _t['tenure_days'] = (_t['last_d'] - _t['first_d']).dt.days
        _t['days_silent'] = (_ref - _t['last_d']).dt.days
        _t['days_per_month'] = _t['days_measured'] / (_t['tenure_days'] / 30.44).clip(lower=1)
        prof.append({'iso3_code': _iso, 'schools': len(_t),
                     'median_tenure_months': _t['tenure_days'].median() / 30.44,
                     'median_days_per_month': _t['days_per_month'].median(),
                     'p25_days_per_month': _t['days_per_month'].quantile(.25),
                     'p75_days_per_month': _t['days_per_month'].quantile(.75),
                     'school_days': int(_t['days_measured'].sum()),
                     'school_years_observed': float((_ref - _t['first_d']).dt.days.sum()) / 365.25,
                     **{f'school_years_at_risk_{_h}':
                        float(((_ref - _t['first_d']).dt.days - _h).clip(lower=0).sum()) / 365.25
                        for _h in (7, 14, 30, 90, 182)},
                     'school_years_at_risk': float(((_ref - _t['first_d']).dt.days - 182)
                                                   .clip(lower=0).sum()) / 365.25,
                     **{f'n_eligible_{_h}d': int(((_ref - _t['first_d']).dt.days >= _h).sum())
                        for _h in (30, 90, 182)},
                     **{f'pct_silent_{_h}d': (100 * (_t['days_silent'] > _h).sum()
                        / max(int(((_ref - _t['first_d']).dt.days >= _h).sum()), 1))
                        for _h in (30, 90, 182)},
                     'data_from': _sd['d'].min(), 'data_to': _ref})
        _g = _sd['gap'].dropna()
        rhy.append({'iso3_code': _iso, 'pct_gap_1d': 100 * (_g == 1).mean(),
                    'pct_gap_2_7d': 100 * _g.between(2, 7).mean(),
                    'pct_gap_8_30d': 100 * _g.between(8, 30).mean(),
                    'pct_gap_over_30d': 100 * (_g > 30).mean(),
                    'pct_weekday': 100 * (_sd['d'].dt.weekday < 5).mean()})
        _sp = pd.concat([_g[_g >= 2].value_counts().rename_axis('dur').reset_index(name='n').assign(returned=1),
                         _t.loc[_t['days_silent'] >= 2, 'days_silent'].value_counts()
                           .rename_axis('dur').reset_index(name='n').assign(returned=0)])
        spl.append(_sp.assign(iso3_code=_iso))
        _exp = pd.DataFrame([(s, p) for s, (a, b) in _t[['first_d', 'last_d']].iterrows()
                             for p in pd.period_range(a.to_period('M'), b.to_period('M'), freq='M')],
                            columns=['school_id_giga', 'm'])
        _am = (_sd.assign(m=_sd['d'].dt.to_period('M')).groupby(['school_id_giga', 'm'])
               .size().rename('days').reset_index())
        _j = _exp.merge(_am, on=['school_id_giga', 'm'], how='left').fillna({'days': 0})
        _s = (_j.assign(calendar_month=_j['m'].dt.month).groupby('calendar_month')
              .agg(mean_days_per_school_month=('days', 'mean'), school_months=('days', 'size')).reset_index())
        sea.append(_s.assign(iso3_code=_iso))
    return (pd.DataFrame(prof), pd.DataFrame(rhy),
            pd.concat(spl, ignore_index=True), pd.concat(sea, ignore_index=True),
            pd.DataFrame())          # first-spell view needs Trino

COUNTRY_ALIASES = {'Fiji': ['FJI'], 'Mongolia': ['MNG'], 'Malawi': ['MWI'],
                   'South Africa': ['ZAF'], 'Uzbekistan': ['UZB'], 'Montenegro': ['MNE'],
                   'Sri Lanka': ['LKA'], 'Kenya': ['KEN'], 'Botswana': ['BWA'],
                   'Rwanda': ['RWA'], 'Zambia': ['ZMB'], 'Ethiopia': ['ETH']}

try:
    if not USE_TRINO:
        raise RuntimeError("USE_TRINO is False")
    profile, rhythm, spells, season, first_spell = _from_trino()
    _mode = "Trino (server-side)"
except Exception as _e:
    print(f"⚠️  falling back to local caches: {_e}")
    profile, rhythm, spells, season, first_spell = _from_cache()
    _mode = "local caches"

profile = profile[profile['schools'] >= MIN_SCHOOLS].set_index('iso3_code')
rhythm = rhythm.set_index('iso3_code')

In [ ]:
print(f"✓ {_mode}: {len(profile)} countries · {len(spells):,} spell rows · {len(season)} seasonal rows")

# `profile` is an intermediate frame — its analytical columns are re-presented in
# the fleet comparison table at the end. What is worth reading HERE is provenance:
# how much data each country has, and how many schools are old enough to be
# judged. Everything else stays on the frame for the calculations downstream.
coverage = pd.DataFrame({
    'schools': profile['schools'].astype(int),
    'data from': pd.to_datetime(profile['data_from']).dt.date,
    'data to': pd.to_datetime(profile['data_to']).dt.date,
    'school-years observed': profile['school_years_observed'].round(0),
    'eligible 30d': profile['n_eligible_30d'].astype(int),
    'eligible 90d': profile['n_eligible_90d'].astype(int),
    'eligible 182d': profile['n_eligible_182d'].astype(int),
    'silent >30d %': profile['pct_silent_30d'].round(0),
    'silent >90d %': profile['pct_silent_90d'].round(0),
}).sort_values('silent >30d %', ascending=False)
print("Coverage and eligibility — how much each country has been watched, and how many of its")
print("schools are old enough for each silence threshold to be observable. 'silent >30d %' uses")
print("the eligible count as its denominator, never the full school list.")
# red = more of the fleet is quiet; the two silence columns share one scale so they
# can be compared against each other as well as across countries
display(coverage.style
        .background_gradient(cmap='RdYlGn_r', subset=['silent >30d %', 'silent >90d %'], vmin=0, vmax=100)
        .format({'school-years observed': '{:,.0f}',
                 'silent >30d %': '{:.0f}%', 'silent >90d %': '{:.0f}%'}))

---
## Part 2 — Seasonality, learned per country

Months are compared with the country's **median** month. A peak-relative rule
mislabels most of the year wherever one month towers over the rest.

In [ ]:
# =============================================================================
# LOW SEASON — published school calendars as the prior, data as confirmation
# =============================================================================
# Inferring the break purely from the data produces false positives: a country
# with one towering peak month puts ordinary months below any relative line
# (Mongolia flagged Oct/Nov, Sri Lanka Nov). So the published school calendar is
# theprior, and the measurement dip either confirms it or flags a mismatch.
#
# Sources (checked Aug 2026 — update as ministries publish):
#   FJI  fiji.gov.fj / Ministry of Education: 7-week break 7 Dec – 22 Jan,
#        2-week breaks early May and late Aug–early Sep
#   MNG  school year from 1 Sep; summer vacation Jun–Aug, winter break ~Jan,
#        additional breaks Nov and late Mar–Apr
#   MWI  2025/26 calendar runs 22 Sep – 24 Jul → long break Aug–mid Sep
#   LKA  three terms Jan–Apr, Apr–Aug, Sep–Dec; breaks Dec–Jan, Apr, Aug
#   ZAF  mid-Jan to early Dec; long break Dec–early Jan, plus Mar/Apr, Jun/Jul, Sep/Oct
#   KAZ  2 Sep – 25 May; summer Jun–Aug; winter break mid-Dec – early Jan
#   UZB/MDA  Sep–May/Jun with a 13–14 week summer (among the longest in Europe)
#   ALB/BIH/MNE  Sep–Jun; summer Jul–Aug; winter break Jan
#   KEN  three terms, extended end-of-year break from late Oct/Nov into Jan
#   RWA  terms Sep–Dec, Jan–Apr, Apr–Jun; long break Jul–Aug
#   BWA/NAM/ZMB  southern-hemisphere year; long break Dec–Jan
#   BLZ/GRD/LCA/TTO/VCT  Sep–Jun; summer Jul–Aug; Christmas Dec
#   HND  Feb–Nov; long break Dec–Jan
#   BEN  Sep–Jul; long break Aug
SCHOOL_HOLIDAY_MONTHS = {
    # --- Pacific / Asia -------------------------------------------------------
    'FJI': [12, 1, 5, 8],        # 7-wk break 7 Dec-22 Jan; 2-wk breaks May, late Aug
    'MNG': [6, 7, 8, 1, 11],     # year from 1 Sep; summer Jun-Aug, winter ~Jan, Nov break
    'LKA': [12, 1, 4, 8],        # terms Jan-Apr, Apr-Aug, Sep-Dec; breaks Dec-Jan, Apr, Aug
    # --- Central Asia / Eastern Europe (Sep-May year, long summer) ------------
    'UZB': [6, 7, 8],            # ~13-14 wk summer; year resumes mid-Sep
    'KAZ': [6, 7, 8, 1],         # 2 Sep-25 May; summer Jun-Aug; winter break late Dec-early Jan
    'MDA': [6, 7, 8],            # among the longest summers in Europe (13-14 wks)
    'ALB': [7, 8],               # Sep-Jun; summer Jul-Aug
    'BIH': [7, 8, 1],            # Sep-Jun; summer Jul-Aug; winter break Jan
    'MNE': [7, 8, 1],            # Sep-Jun; summer Jul-Aug; winter break Jan
    # --- Africa ---------------------------------------------------------------
    'ZAF': [12, 1, 4, 7, 10],    # mid-Jan to early Dec; breaks Dec-Jan, Mar/Apr, Jun/Jul, Sep/Oct
    'MWI': [8, 9],               # 2025/26 year 22 Sep-24 Jul -> long break Aug-mid Sep
    'BWA': [12, 1, 4, 8],        # southern-hemisphere year; long break Dec-Jan
    'NAM': [12, 1, 5, 8],        # southern-hemisphere year; long break Dec-Jan
    'ZMB': [12, 1, 4, 8],        # terms Jan-Apr, May-Aug, Sep-Dec; long break Dec-Jan
    'KEN': [12, 1, 4, 8],        # three terms; extended end-of-year break Nov/Dec-Jan
    'RWA': [7, 8, 12],           # terms Sep-Dec, Jan-Apr, Apr-Jun; long break Jul-Aug
    'BEN': [8, 7],               # Sep-Jul year; long break Aug (Jul from mid-month)
    # --- Caribbean / Central America (Sep-Jun year) --------------------------
    'BLZ': [7, 8, 12],           # Sep-Jun; summer Jul-Aug; Christmas break Dec
    'GRD': [7, 8, 12],
    'LCA': [7, 8, 12],
    'TTO': [7, 8, 12],
    'VCT': [7, 8, 12],
    'HND': [12, 1],              # Feb-Nov year; long break Dec-Jan
}

_MON = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
_names = lambda ms: ', '.join(_MON[m - 1] for m in sorted(ms)) or '—'

LOW_SEASON, _rows = {}, []
for _iso, _g in season.groupby('iso3_code'):
    if _iso not in profile.index:
        continue
    _s = _g.set_index('calendar_month')['mean_days_per_school_month'].reindex(range(1, 13))
    _dip = set(_s[_s < LOW_SEASON_FRAC * _s.median()].dropna().index)      # data says quiet
    _pub = set(SCHOOL_HOLIDAY_MONTHS.get(_iso, []))                        # calendar says break
    if _pub:
        _confirmed = sorted(_dip & _pub)          # break AND quiet -> trust
        _source = 'calendar ∩ data'
        LOW_SEASON[_iso] = _confirmed or sorted(_pub)
    else:
        _confirmed = sorted(_dip)
        _source = 'data only (no calendar on file)'
        LOW_SEASON[_iso] = _confirmed
    _span = (pd.Timestamp(profile.loc[_iso, 'data_to']) - pd.Timestamp(profile.loc[_iso, 'data_from'])).days / 30.44
    _rows.append({'iso3_code': _iso,
                  'published break': _names(_pub) if _pub else '—',
                  'data dip': _names(_dip),
                  'LOW SEASON used': _names(LOW_SEASON[_iso]),
                  'basis': _source,
                  'dip not in calendar': _names(_dip - _pub) if _pub else '—',
                  'break without dip': _names(_pub - _dip) if _pub else '—',
                  'trough % of median': round(100 * _s.min() / _s.median()),
                  'span (mo)': round(_span),
                  'seasonality': 'provisional (<2 yrs)' if _span < MIN_SPAN_MONTHS_FOR_SEASON else 'established'})
season_summary = pd.DataFrame(_rows).set_index('iso3_code')
display(season_summary)
print("'dip not in calendar' = quiet months the school calendar does not explain (investigate: "
      "device or\nengagement problem, not a holiday). 'break without dip' = schools kept measuring "
      "through a break\n(often boarding schools or devices left on).")

_order = list(profile.index)
_ncol = 4
_nrow = int(np.ceil(len(_order) / _ncol))
fig, axes = plt.subplots(_nrow, _ncol, figsize=(4.2 * _ncol, 2.6 * _nrow), squeeze=False)
_flat = axes.ravel()
for _ax in _flat[len(_order):]:
    _ax.set_visible(False)
for _ax, _iso in zip(_flat, _order):
    _s = season[season['iso3_code'] == _iso].set_index('calendar_month')['mean_days_per_school_month'].reindex(range(1, 13))
    _pub = set(SCHOOL_HOLIDAY_MONTHS.get(_iso, []))
    _cols = [GIGA_MODERATE if c in LOW_SEASON[_iso] else
             (GIGA_PRIMARY[200] if c in _pub else GIGA_PRIMARY[600]) for c in range(1, 13)]
    _ax.bar(range(1, 13), _s.values, color=_cols, edgecolor='white')
    _ax.axhline(LOW_SEASON_FRAC * _s.median(), color=GIGA_GREY[700], ls='--', lw=1)
    _ax.set_xticks(range(1, 13))
    _ax.set_xticklabels([m[0] for m in _MON], fontsize=7)
    _ax.set_title(f'{_iso}: {_names(LOW_SEASON[_iso])}', fontsize=9)
    _ax.tick_params(axis='y', labelsize=7)
    _ax.set_ylabel('days/school-mo', fontsize=7)
fig.suptitle('Seasonal rhythm — amber = low season used · pale blue = published break with no dip', y=1.005)
plt.tight_layout(); plt.show()

---
## Part 3 — Silence and churn

Kaplan–Meier per country on the spell histogram: conditional on a school already
being silent X days, how likely is it to return? Censored spells (schools quiet
right now) stay in the risk set instead of counting as failures.

In [ ]:
# =============================================================================
# CONDITIONAL SURVIVAL + CHURN INCIDENCE, PER COUNTRY
# =============================================================================
SILENCE_POINTS = [7, 14, 21, 30, 45, 60, 90, 182]
HORIZONS = [7, 14, 30, 90]

def _km(df):
    """Kaplan-Meier survivor from a (dur, returned, n) histogram."""
    _d = df.groupby(['dur', 'returned'])['n'].sum().unstack(fill_value=0)
    _d = _d.reindex(columns=[0, 1], fill_value=0).sort_index()
    _at_risk = _d.sum(axis=1)[::-1].cumsum()[::-1]
    _S, _out = 1.0, {}
    for _t, _row in _d.iterrows():
        if _row[1] > 0:
            _S *= (1 - _row[1] / _at_risk[_t])
        _out[_t] = _S
    return pd.Series(_out)

CHURN_HORIZONS = [7, 14, 30, 90, 182]   # 'churned' at each definition of silence
# NOTE 7d and 14d are NOT churn in any real sense — 88% of pauses are <=7 days and
# ~93% of schools silent 7 days come back. They are included so the rate can be read
# as a gradient: the drop from @7d to @182d is how much silence is temporary.
MIN_AT_RISK_YEARS = 20             # below this, the rate is not reportable

surv_tables, churn_rows = {}, []
for _iso in profile.index:
    _K = _km(spells[spells['iso3_code'] == _iso])
    _surv = lambda x, _K=_K: (_K[_K.index <= x].iloc[-1] if len(_K[_K.index <= x]) else 1.0)
    _sp_i = spells[spells['iso3_code'] == _iso]
    _t = pd.DataFrame({'silent_for_days': SILENCE_POINTS})
    _t['risk set'] = [int(_sp_i.loc[_sp_i['dur'] >= X, 'n'].sum()) for X in SILENCE_POINTS]
    _t['still silent'] = [int(_sp_i.loc[(_sp_i['dur'] >= X) & (_sp_i['returned'] == 0), 'n'].sum())
                          for X in SILENCE_POINTS]
    for _h in HORIZONS:
        _t[f'+{_h}d %'] = [round(100 * (1 - _surv(X + _h) / _surv(X))) for X in SILENCE_POINTS]
    _t['by 365d % (spells)'] = [round(100 * (1 - _surv(365) / _surv(X))) for X in SILENCE_POINTS]
    # School-level companion: one spell per school (its FIRST pause), so a school
    # that pauses every December is counted once instead of dozens of times. The
    # spell view answers "of pauses this long, how many end?"; this one answers
    # "of SCHOOLS that go this quiet, how many come back?".
    if len(first_spell):
        _fs = first_spell[first_spell['iso3_code'] == _iso]
        if len(_fs):
            _Kf = _km(_fs)
            _survf = lambda x, _Kf=_Kf: (_Kf[_Kf.index <= x].iloc[-1] if len(_Kf[_Kf.index <= x]) else 1.0)
            _t['by 365d % (schools)'] = [round(100 * (1 - _survf(365) / _survf(X))) for X in SILENCE_POINTS]
            _t['schools at X'] = [int(_fs.loc[_fs['dur'] >= X, 'n'].sum()) for X in SILENCE_POINTS]
    surv_tables[_iso] = _t.set_index('silent_for_days')

    # churn incidence at each horizon. Exposure MUST match the definition: a
    # silence of N days is only observable in school-time that began >= N days
    # before the last data date — otherwise young fleets score a fake zero.
    _row = {'iso3_code': _iso,
            'schools': int(profile.loc[_iso, 'schools']),
            'schools silent now >30d': int(round(profile.loc[_iso, 'pct_silent_30d']
                                                 * profile.loc[_iso, 'n_eligible_30d'] / 100)),
            'schools silent now >90d': int(round(profile.loc[_iso, 'pct_silent_90d']
                                                 * profile.loc[_iso, 'n_eligible_90d'] / 100))}
    for _h in CHURN_HORIZONS:
        _events = int(_sp_i[(_sp_i['dur'] >= _h) & (_sp_i['returned'] == 0)]['n'].sum())
        _risk = float(profile.loc[_iso, f'school_years_at_risk_{_h}'])
        _row[f'silent>{_h}d (n)'] = _events
        _row[f'yrs at risk {_h}d'] = round(_risk)
        _row[f'churn/100 yrs @{_h}d'] = round(100 * _events / _risk, 1) if _risk >= MIN_AT_RISK_YEARS else np.nan
    _row['P(return | 30d) %'] = int(surv_tables[_iso].loc[30, 'by 365d % (spells)'])
    _row['P(return | 90d) %'] = int(surv_tables[_iso].loc[90, 'by 365d % (spells)'])
    churn_rows.append(_row)

churn = pd.DataFrame(churn_rows).set_index('iso3_code')
churn = churn.sort_values('churn/100 yrs @182d', ascending=False, na_position='last')
print("CHURN INCIDENCE at three definitions of 'gone' — events per 100 school-years AT RISK\n")
print("  silent>Nd (n)      schools whose CURRENT silence has already run N+ days (still silent")
print("                     at the last data date) — i.e. observed churn events")
print("  yrs at risk Nd     school-time in which such an event COULD have been seen: for each")
print("                     school, (days from its first measurement to the country's last data")
print("                     date) minus N, floored at zero, summed and divided by 365.25. A school")
print("                     that started 60 days ago contributes nothing to the 90-day column,")
print("                     because it cannot yet have been silent that long.")
print("  churn/100 yrs @Nd  events / at-risk years x 100 — the exposure-matched rate, comparable")
print("                     across countries and across fleet ages. Blank = too little exposure.\n")
_rate_cols = [f'churn/100 yrs @{_h}d' for _h in CHURN_HORIZONS]
_show = ['schools', 'schools silent now >30d', 'schools silent now >90d'] + \
        [c for _h in CHURN_HORIZONS for c in (f'silent>{_h}d (n)', f'churn/100 yrs @{_h}d')]
_vmax = float(np.nanpercentile(churn[_rate_cols].to_numpy(dtype=float), 95))
display(churn[_show].style
        .background_gradient(cmap='RdYlGn_r', subset=_rate_cols, vmin=0, vmax=_vmax)
        .format({c: '{:.1f}' for c in _rate_cols}, na_rep='—'))
print("(at-risk exposure columns are used in the rates and omitted here; "
      "see 'yrs at risk' in the churn frame)")
_thin = churn[churn['churn/100 yrs @182d'].isna()]
if len(_thin):
    print(f"no 182-day rate (under {MIN_AT_RISK_YEARS} school-years at risk): {', '.join(_thin.index)} — "
          f"too young for the event to be observable, which is NOT zero churn. Their 30- and 90-day "
          f"rates are still valid.")

fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * len(churn))))
_p = churn.dropna(subset=['churn/100 yrs @182d']).sort_values('churn/100 yrs @182d')
_y = np.arange(len(_p)); _h = 0.26
for _k, (_hz, _c) in enumerate(zip(CHURN_HORIZONS, [GIGA_PRIMARY[200], GIGA_PRIMARY[500], GIGA_PRIMARY[800]])):
    ax.barh(_y + (_k - 1) * _h, _p[f'churn/100 yrs @{_hz}d'], height=_h, color=_c, label=f'silent >{_hz}d')
ax.set_yticks(_y); ax.set_yticklabels(_p.index, fontsize=8)
ax.set_xlabel('churn events per 100 school-years at risk')
ax.set_title('Churn incidence by definition of "gone"')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for _iso, _t in surv_tables.items():
    print(f"\n{_iso} — given a school is already silent for X days, % that return."
          f"\n  risk set = silence spells that actually reached X days (completed + still running);"
          f"\n  still silent = of those, the ones ongoing at the last data date (censored, not failures)."
          f"\n  'by 365d % (spells)' counts EPISODES — a school that pauses every year contributes one"
          f"\n  each time. 'by 365d % (schools)' counts each school once, using its FIRST pause, and"
          f"\n  'schools at X' is that school count. Quote the school column to country teams."
          f"\n  Rows with a risk set under ~30 are unstable.")
    display(_t)

# ── Comeback curves: grey context + labeled highlights (a 22-line hue-cycled
# legend is unreadable; identity via direct labels, not color) ─────────────────
_cb = pd.DataFrame({_iso: _t['by 365d % (spells)'] for _iso, _t in surv_tables.items()
                    if _t.loc[30, 'risk set'] >= 30})   # unstable curves excluded
_dropped = sorted(set(surv_tables) - set(_cb.columns))
_at90 = _cb.loc[90]
_INK = '#3a3f46'
_hi = {_at90.idxmax(): (GIGA_PRIMARY[800], 'most comebacks'),
       _at90.idxmin(): ('#c0392b', 'fewest comebacks')}
_big = max(surv_tables, key=lambda k: surv_tables[k].loc[7, 'risk set'])
if _big in _cb.columns and _big not in _hi:
    _hi[_big] = (GIGA_PRIMARY[500], 'largest fleet')

fig, ax = plt.subplots(figsize=(10, 5))
for _iso in _cb.columns:
    if _iso not in _hi:
        ax.plot(_cb.index, _cb[_iso], color='#c9ced6', lw=1, zorder=1)
_med = _cb.median(axis=1)
ax.plot(_med.index, _med, color=_INK, lw=2, ls='--', zorder=2)
for _iso, (_col, _why) in _hi.items():
    ax.plot(_cb.index, _cb[_iso], color=_col, lw=2.4, marker='o', ms=5, zorder=3)
# end labels: clamp into the axis and push apart so none collide
_x_end = _cb.index[-1]
_labels = [[_med.iloc[-1], 'fleet median', 'normal']] + \
          [[_cb.loc[_x_end, _iso], f'{_iso} — {_why}', 'bold'] for _iso, (_c2, _why) in _hi.items()]
_labels.sort(key=lambda l: l[0])
_MINGAP = 5.0
for _k in range(len(_labels)):
    _labels[_k][0] = max(_labels[_k][0], 3.0)
    if _k and _labels[_k][0] - _labels[_k - 1][0] < _MINGAP:
        _labels[_k][0] = _labels[_k - 1][0] + _MINGAP
for _y, _txt, _wt in _labels:
    ax.annotate(_txt, (_x_end, min(_y, 98.0)), xytext=(7, 0), textcoords='offset points',
                va='center', fontsize=9, fontweight=_wt, color=_INK)
ax.set_xlabel('days already silent'); ax.set_ylabel('% returning within a year')
ax.set_title('Silence is not the same thing in every country')
ax.set_xlim(0, _cb.index[-1] * 1.28); ax.set_ylim(0, 100)
_note = 'grey = all other countries'
if _dropped:
    _note += f' · excluded (<30 spells at 30d): {", ".join(_dropped)}'
ax.text(0.0, -0.18, _note, transform=ax.transAxes, fontsize=8, color='#6f6f6f')
plt.tight_layout(); plt.show()


---
## Part 3b — Campaign control

Returns are not always organic. Where a reactivation campaign ran, the return
spike belongs to the campaign, and the country's headline numbers describe a
fleet under intervention.

In [ ]:
# =============================================================================
# RETURN EVENTS BY MONTH — so campaign effects are visible, not baked in
# =============================================================================
Q_RETURNS_MONTHLY = BASE_CTE + """
SELECT iso3_code,
       date_trunc('month', d) AS return_month,
       count_if(dur BETWEEN 31 AND 90)  AS returns_31_90d,
       count_if(dur > 90)               AS returns_over_90d,
       count(*)                         AS returns_over_30d
FROM (
    SELECT iso3_code, d, date_diff('day', prev_d, d) AS dur
    FROM lagged WHERE prev_d IS NOT NULL
) WHERE dur > 30
GROUP BY 1, 2
"""

def _returns_from_cache():
    _out = []
    for _dir in sorted(CACHE_ROOT.iterdir()):
        _f = sorted(_dir.glob('*_measurements.parquet')) if _dir.is_dir() else []
        if not _f:
            continue
        _iso = next((c for c in COUNTRY_ALIASES.get(_dir.name, []) if not COUNTRIES or c in COUNTRIES), None)
        if _iso is None or (COUNTRIES and _iso not in COUNTRIES):
            continue
        _m = pd.read_parquet(_f[0], columns=['school_id_giga', 'date'])
        _m['d'] = pd.to_datetime(_m['date'], errors='coerce', utc=True).dt.tz_localize(None).dt.normalize()
        _sd = _m.dropna(subset=['d']).drop_duplicates(['school_id_giga', 'd']).sort_values(['school_id_giga', 'd'])
        _sd['dur'] = _sd.groupby('school_id_giga')['d'].diff().dt.days
        _r = _sd[_sd['dur'] > 30].copy()
        _r['return_month'] = _r['d'].dt.to_period('M').dt.to_timestamp()
        _g = _r.groupby('return_month').agg(
            returns_31_90d=('dur', lambda s: int(s.between(31, 90).sum())),
            returns_over_90d=('dur', lambda s: int((s > 90).sum())),
            returns_over_30d=('dur', 'size')).reset_index()
        _out.append(_g.assign(iso3_code=_iso))
    return pd.concat(_out, ignore_index=True) if _out else pd.DataFrame()

try:
    if not USE_TRINO:
        raise RuntimeError('USE_TRINO is False')
    _cur = get_trino_cursor()
    _cur.execute(Q_RETURNS_MONTHLY)
    returns_monthly = pd.DataFrame(_cur.fetchall(), columns=[c[0] for c in _cur.description])
except Exception as _e:
    print(f"⚠️  returns-by-month from local caches: {_e}")
    returns_monthly = _returns_from_cache()
returns_monthly['return_month'] = pd.to_datetime(returns_monthly['return_month'])

_order = list(profile.index)
_ncol = 4
_nrow = int(np.ceil(len(_order) / _ncol))
fig, axes = plt.subplots(_nrow, _ncol, figsize=(4.4 * _ncol, 2.7 * _nrow), squeeze=False)
_flat = axes.ravel()
for _ax in _flat[len(_order):]:
    _ax.set_visible(False)
for _ax, _iso in zip(_flat, _order):
    _g = returns_monthly[returns_monthly['iso3_code'] == _iso].sort_values('return_month')
    _ax.bar(_g['return_month'], _g['returns_over_30d'], width=20, color=GIGA_PRIMARY[600])
    for _a, _b, _lbl in campaign_windows(_iso):
        _ax.axvspan(_a, _b or _g['return_month'].max(), color=GIGA_MODERATE, alpha=0.30)
        _ax.text(_a, _ax.get_ylim()[1] * 0.92, ' campaign', fontsize=8, color=GIGA_GREY[800])
    for _m in LOW_SEASON.get(_iso, []):
        for _yr in _g['return_month'].dt.year.unique():
            _ax.axvline(pd.Timestamp(year=int(_yr), month=int(_m), day=15), color=GIGA_GREY[300], lw=6, alpha=.25)
    _ax.set_title(f'{_iso}', fontsize=9)
    _ax.tick_params(axis='x', rotation=45, labelsize=6)
    _ax.tick_params(axis='y', labelsize=7)
    _ax.set_ylabel('returns', fontsize=7)
fig.suptitle('Schools returning after >30d silence, by month (amber = campaign window, grey = low season)', y=1.005)
plt.tight_layout(); plt.show()

# how much of each country's return volume sits inside a campaign window
_rows = []
for _iso in profile.index:
    _g = returns_monthly[returns_monthly['iso3_code'] == _iso]
    if _g.empty:
        continue
    _inw = _g[[in_campaign(_iso, t) for t in _g['return_month']]]
    _rows.append({'iso3_code': _iso, 'returns >30d (total)': int(_g['returns_over_30d'].sum()),
                  'inside campaign window': int(_inw['returns_over_30d'].sum()),
                  '% campaign-driven': round(100 * _inw['returns_over_30d'].sum() /
                                             max(_g['returns_over_30d'].sum(), 1), 1),
                  'campaign in window': bool(CAMPAIGNS.get(_iso))})
campaign_effect = pd.DataFrame(_rows).set_index('iso3_code')
display(campaign_effect)
print("Where '% campaign-driven' is material, the country's return probabilities and churn rate")
print("describe a fleet UNDER INTERVENTION — compare it with its own pre-campaign baseline, not")
print("with countries that had no campaign.")

---
## Part 3c — Age-aligned comparison

Churn rates are not comparable between a four-year-old fleet and a
six-month-old one. Aligning on school tenure removes fleet age from the
comparison and is the control that a "provisional" flag only approximates.


In [ ]:
# =============================================================================
# TENURE-ALIGNED SURVIVAL — compare fleets at the same AGE, not the same date
# =============================================================================
# Countries started years apart, so raw churn compares a 4-year-old fleet with a
# 6-month-old one. Aligning on tenure removes that: for every school, how long
# did it keep measuring after its FIRST measurement? Each age column counts only
# schools with at least that much follow-up in their own country, so a young
# fleet contributes to the columns it has earned and no others.
AGES = [3, 6, 12, 18, 24]

_age_sql = ",\n       ".join(
    f"count_if(followup_m >= {a}) AS n_age{a}, "
    f"round(100.0 * count_if(followup_m >= {a} AND lifespan_m >= {a}) "
    f"/ NULLIF(count_if(followup_m >= {a}), 0), 0) AS alive_{a}mo" for a in AGES)
Q_TENURE = f"""
WITH sd AS (
    SELECT iso3_code, school_id_giga, CAST({_LTS} AS date) AS d
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_scope_sql}
    GROUP BY 1, 2, 3
),
ten AS (SELECT iso3_code, school_id_giga, min(d) AS first_d, max(d) AS last_d FROM sd GROUP BY 1, 2),
ref AS (SELECT iso3_code, max(d) AS ref_d FROM sd GROUP BY 1),
elig AS (
    SELECT t.iso3_code, t.school_id_giga,
           date_diff('month', t.first_d, r.ref_d)  AS followup_m,
           date_diff('month', t.first_d, t.last_d) AS lifespan_m
    FROM ten t JOIN ref r ON r.iso3_code = t.iso3_code
)
SELECT iso3_code,
       {_age_sql}
FROM elig GROUP BY 1
HAVING count_if(followup_m >= 6) >= 30
ORDER BY alive_12mo DESC NULLS LAST
"""

try:
    _cur = get_trino_cursor()
    _cur.execute(Q_TENURE)
    tenure_survival = pd.DataFrame(_cur.fetchall(), columns=[c[0] for c in _cur.description]).set_index('iso3_code')
    print("% of schools still measuring N months after their first measurement "
          "(n = schools with that much follow-up):")
    display(tenure_survival)

    _plot = tenure_survival[[f'alive_{a}mo' for a in AGES]].astype(float)
    _plot = _plot.mask(tenure_survival[[f'n_age{a}' for a in AGES]].astype(float).lt(30).values)
    _big = tenure_survival['n_age12'].astype(float).fillna(0) >= 50
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for _iso, _row in _plot[_big].iterrows():
        ax.plot(AGES, _row.values, marker='o', label=_iso)
    ax.set_xticks(AGES); ax.set_xlabel('months since first measurement'); ax.set_ylim(0, 100)
    ax.set_ylabel('% of schools still measuring')
    ax.set_title('Fleet survival at the same age (countries with >= 50 schools at 12 months)')
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout(); plt.show()
    # FLEET total — pooled across every country, plus a fleet median of country rates
    _fleet_row = {}
    for _a in AGES:
        _n = tenure_survival[f'n_age{_a}'].astype(float)
        _alive = tenure_survival[f'alive_{_a}mo'].astype(float)
        _valid = _n.notna() & _alive.notna() & (_n > 0)
        _fleet_row[f'n_age{_a}'] = int(_n[_valid].sum())
        _fleet_row[f'alive_{_a}mo'] = round(float((_n[_valid] * _alive[_valid]).sum() / _n[_valid].sum()), 0)
    _median_row = {f'n_age{_a}': int(tenure_survival[f'n_age{_a}'].astype(float).gt(0).sum()) for _a in AGES}
    _median_row.update({f'alive_{_a}mo': round(float(
        tenure_survival.loc[tenure_survival[f'n_age{_a}'].astype(float) >= 30, f'alive_{_a}mo'].astype(float).median()), 0)
        for _a in AGES})
    survivorship = pd.concat([
        tenure_survival,
        pd.DataFrame([_fleet_row, _median_row], index=['FLEET (pooled schools)', 'FLEET (median country, n>=30)'])])
    print("\nSURVIVORSHIP — % of schools still measuring at 3 / 6 / 12 / 18 / 24 months")
    display(survivorship[[c for _a in AGES for c in (f'n_age{_a}', f'alive_{_a}mo')]])

    _exp = CACHE_ROOT / '_fleet'
    _exp.mkdir(exist_ok=True)
    survivorship.to_csv(_exp / 'fleet_survivorship.csv')
    print(f"exported {_exp / 'fleet_survivorship.csv'}")
    print("Read down a column, never across a row: a country missing alive_24mo simply has not")
    print("existed that long. Small n at older ages is also a survivorship sample — the schools")
    print("with 24 months of follow-up are early adopters, who are not typical of later cohorts.")
except Exception as _e:
    tenure_survival = pd.DataFrame()
    print(f"(tenure-aligned survival needs Trino: {_e})")

---
## Part 3d — App version of the live fleet

Which still-measuring schools are on an old build. Restricted to active
schools deliberately: an abandoned school on an old version is a churn
problem, not an upgrade problem.


In [ ]:
# =============================================================================
# APP VERSION OF SCHOOLS STILL MEASURING — who is stuck on an old build?
# =============================================================================
# Version is taken from each school's MOST RECENT measurement, and only schools
# still measuring (silent <= ACTIVE_DAYS) are counted — an abandoned school on
# 1.0.4 is a churn problem, not an upgrade problem.
# 1.0.9 is the cut asked for: below it the app predates the 2024-11 line, and
# device_id (2.0.2+) and several later fields are unavailable.
ACTIVE_DAYS = 30
OLD_BELOW = '1.0.9'

Q_APPVER = f"""
WITH last_meas AS (
    -- local date everywhere, to match BASE_CTE (UTC `date` differs on 4.3% of rows)
    SELECT iso3_code, school_id_giga, app_version,
           CAST({_LTS} AS date) AS d,
           ROW_NUMBER() OVER (PARTITION BY iso3_code, school_id_giga
                              ORDER BY CAST({_LTS} AS date) DESC) AS rn
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_scope_sql}
),
ref AS (SELECT iso3_code, max(CAST({_LTS} AS date)) AS ref_d
        FROM default.all_gigameter_measurement_data
        WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_scope_sql} GROUP BY 1),
cur AS (
    SELECT l.iso3_code, l.school_id_giga, l.app_version,
           date_diff('day', l.d, r.ref_d) AS days_silent
    FROM last_meas l JOIN ref r ON r.iso3_code = l.iso3_code
    WHERE l.rn = 1
)
SELECT iso3_code,
       count_if(days_silent <= {ACTIVE_DAYS}) AS active_schools,
       count_if(days_silent <= {ACTIVE_DAYS} AND
                (app_version IS NULL OR
                 CAST(split_part(app_version,'.',1) AS INTEGER) < 1 OR
                 (CAST(split_part(app_version,'.',1) AS INTEGER) = 1 AND
                  CAST(split_part(app_version,'.',2) AS INTEGER) = 0 AND
                  CAST(split_part(app_version,'.',3) AS INTEGER) < 9))) AS active_below_1_0_9,
       count_if(days_silent <= {ACTIVE_DAYS} AND
                CAST(split_part(app_version,'.',1) AS INTEGER) >= 2) AS active_on_2x,
       count_if(days_silent <= {ACTIVE_DAYS} AND app_version = '2.0.3') AS active_on_latest
FROM cur GROUP BY 1 HAVING count_if(days_silent <= {ACTIVE_DAYS}) > 0
ORDER BY 3 DESC
"""

try:
    _cur = get_trino_cursor()
    _cur.execute(Q_APPVER)
    appver = pd.DataFrame(_cur.fetchall(), columns=[c[0] for c in _cur.description]).set_index('iso3_code')
    appver['% below 1.0.9'] = (100 * appver['active_below_1_0_9'] / appver['active_schools']).round(0)
    appver['% on 2.x'] = (100 * appver['active_on_2x'] / appver['active_schools']).round(0)
    appver['% on 2.0.3'] = (100 * appver['active_on_latest'] / appver['active_schools']).round(0)
    _tot = {'active_schools': int(appver['active_schools'].sum()),
            'active_below_1_0_9': int(appver['active_below_1_0_9'].sum()),
            'active_on_2x': int(appver['active_on_2x'].sum()),
            'active_on_latest': int(appver['active_on_latest'].sum())}
    _tot['% below 1.0.9'] = round(100 * _tot['active_below_1_0_9'] / _tot['active_schools'])
    _tot['% on 2.x'] = round(100 * _tot['active_on_2x'] / _tot['active_schools'])
    _tot['% on 2.0.3'] = round(100 * _tot['active_on_latest'] / _tot['active_schools'])
    appver = pd.concat([appver, pd.DataFrame([_tot], index=['FLEET'])])
    print(f"Schools measuring in the last {ACTIVE_DAYS} days, by app version of their latest test:")
    display(appver[['active_schools', 'active_below_1_0_9', '% below 1.0.9', '% on 2.x', '% on 2.0.3']])

    _p = appver.drop(index='FLEET')
    _p = _p[_p['active_schools'] >= 20].sort_values('% below 1.0.9', ascending=True)
    fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * len(_p))))
    ax.barh(_p.index, _p['% below 1.0.9'], color=GIGA_BAD, label=f'< {OLD_BELOW}')
    ax.barh(_p.index, _p['% on 2.x'], left=_p['% below 1.0.9'], color=GIGA_GOOD, label='2.x')
    ax.set_xlabel('% of still-measuring schools'); ax.set_xlim(0, 100)
    ax.set_title(f'App version of schools still measuring (last {ACTIVE_DAYS} days)')
    ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout(); plt.show()
    print(f"FLEET: {_tot['active_below_1_0_9']:,} of {_tot['active_schools']:,} still-measuring schools "
          f"({_tot['% below 1.0.9']:.0f}%) run a build older than {OLD_BELOW}.")
    print("These are reachable — they are measuring right now — so they are the addressable "
          "upgrade\nqueue, unlike silent schools which need reactivation first.")
except Exception as _e:
    appver = pd.DataFrame()
    print(f"(app-version breakdown needs Trino: {_e})")

---
## Part 3e — Time-of-day coverage

The app schedules one automatic test in each of three 4-hour windows —
morning 08:00–12:00, midday 12:00–16:00, afternoon 16:00–20:00 local
(Giga Meter Scheduler, v2.0.3). This asks how often a school actually
lands all three, where the misses fall, and whether countries concentrate
their measuring at the same times of day.


In [ ]:
# =============================================================================
# SLOT COVERAGE — does a school measure across the whole day the app plans for?
# =============================================================================
# Slots are the app's own scheduling windows (Giga Meter Scheduler doc, v2.0.3):
# while the app is open it runs one automatic test per window, at a random minute
# inside it. Windows are local time and 4 hours long.
#     morning 08:00-12:00 · midday 12:00-16:00 · afternoon 16:00-20:00
# A device on all day should hit all three. Two caveats: the daily launch test
# fires 0-15 min after the app opens, so a morning hit may be a launch test
# rather than the scheduled one; and a failed test retries only 3x at 15-min
# spacing, so a window can be lost to an outage that clears later inside it.
# Hours come from _LTS (timezone-integrity cell), never the raw local column.
SLOT_BOUNDS = [8, 12, 16, 20]      # per the scheduler doc
SLOT_MIN_DAYS = 200                # skip countries with fewer qualifying school-days
SLOT_WEEKDAYS_ONLY = True          # the schedule runs daily; school days are the fair test

_s1, _s2, _s3, _s4 = SLOT_BOUNDS
_dow_sql = f"AND day_of_week(CAST({_LTS} AS date)) <= 5" if SLOT_WEEKDAYS_ONLY else ""

_DAY_SLOTS = f"""
    SELECT iso3_code, school_id_giga, CAST({_LTS} AS date) AS d,
           max(IF(hour({_LTS}) >= {_s1} AND hour({_LTS}) < {_s2}, 1, 0)) AS slot1,
           max(IF(hour({_LTS}) >= {_s2} AND hour({_LTS}) < {_s3}, 1, 0)) AS slot2,
           max(IF(hour({_LTS}) >= {_s3} AND hour({_LTS}) < {_s4}, 1, 0)) AS slot3
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL}
      AND created_timestamp IS NOT NULL
      AND hour({_LTS}) >= {_s1} AND hour({_LTS}) < {_s4}
      {_dow_sql} {_scope_sql}
    GROUP BY 1, 2, 3
"""

Q_SLOTS = f"""
WITH day_slots AS ({_DAY_SLOTS})
SELECT iso3_code,
       count(DISTINCT school_id_giga) AS schools,
       count(*) AS school_days,
       100.0 * avg(IF(slot1 = 1 AND slot2 = 1 AND slot3 = 1, 1e0, 0e0)) AS pct_days_all3,
       100.0 * avg(IF(slot1 = 1 AND slot3 = 1 AND slot2 = 0, 1e0, 0e0)) AS pct_days_1and3_not2,
       100.0 * avg(CAST(slot1 AS DOUBLE)) AS pct_days_morning,
       100.0 * avg(CAST(slot2 AS DOUBLE)) AS pct_days_midday,
       100.0 * avg(CAST(slot3 AS DOUBLE)) AS pct_days_afternoon,
       100.0 * avg(IF(slot1 + slot2 + slot3 = 1, 1e0, 0e0)) AS pct_days_one_slot_only
FROM day_slots GROUP BY 1
HAVING count(*) >= {SLOT_MIN_DAYS}
ORDER BY 4 DESC
"""

Q_SLOTS_SCHOOL = f"""
WITH day_slots AS ({_DAY_SLOTS})
SELECT iso3_code, school_id_giga, count(*) AS school_days,
       100.0 * avg(IF(slot1 = 1 AND slot2 = 1 AND slot3 = 1, 1e0, 0e0)) AS pct_all3,
       100.0 * avg(IF(slot1 = 1 AND slot3 = 1 AND slot2 = 0, 1e0, 0e0)) AS pct_1and3_not2
FROM day_slots GROUP BY 1, 2 HAVING count(*) >= 20
"""

Q_SLOT_HOURS = f"""
SELECT iso3_code, hour({_LTS}) AS hr, count(*) AS n
FROM default.all_gigameter_measurement_data
WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL}
  AND created_timestamp IS NOT NULL
  {_dow_sql} {_scope_sql}
GROUP BY 1, 2
"""

try:
    _cur = get_trino_cursor()
    def _q(sql):
        _cur.execute(sql)
        return pd.DataFrame(_cur.fetchall(), columns=[c[0] for c in _cur.description])
    slots = _q(Q_SLOTS).set_index('iso3_code')
    slots_school = _q(Q_SLOTS_SCHOOL)
    slot_hours = _q(Q_SLOT_HOURS)

    print(f"Scheduler windows (local time{', weekdays only' if SLOT_WEEKDAYS_ONLY else ''}): "
          f"morning {_s1:02d}-{_s2:02d} · midday {_s2:02d}-{_s3:02d} · afternoon {_s3:02d}-{_s4:02d}.")
    print(f"Denominator = school-days with at least one measurement between {_s1:02d}:00 and "
          f"{_s4:02d}:00, i.e. days the device was demonstrably awake.\n")
    _show = ['schools', 'school_days', 'pct_days_all3', 'pct_days_1and3_not2',
             'pct_days_morning', 'pct_days_midday', 'pct_days_afternoon', 'pct_days_one_slot_only']
    _pcts = [c for c in _show if c.startswith('pct')]
    display(slots[_show].round(1).style
            .background_gradient(cmap='RdYlGn', subset=['pct_days_all3'])
            .background_gradient(cmap='RdYlGn_r', subset=['pct_days_one_slot_only'])
            .format('{:,.0f}', subset=['schools', 'school_days'])
            .format('{:.1f}', subset=_pcts))

    _w = slots['school_days']
    _fleet = {c: float((slots[c] * _w).sum() / _w.sum()) for c in _pcts}
    print(f"FLEET ({int(slots['schools'].sum()):,} schools, {int(_w.sum()):,} school-days): "
          f"all three windows on {_fleet['pct_days_all3']:.1f}% of school-days · "
          f"morning+afternoon but not midday on {_fleet['pct_days_1and3_not2']:.1f}% · "
          f"a single window only on {_fleet['pct_days_one_slot_only']:.1f}%")
    print(f"        window hit-rates: morning {_fleet['pct_days_morning']:.1f}% · "
          f"midday {_fleet['pct_days_midday']:.1f}% · afternoon {_fleet['pct_days_afternoon']:.1f}%")

    # Do countries genuinely differ, or is this noise? The school is the unit.
    _elig = slots_school[slots_school['iso3_code'].isin(slots.index)]
    _om = kruskal_omnibus(_elig, unit_col='school_id_giga', group_col='iso3_code',
                          value_col='pct_all3', min_per_cell=1)
    _eff = _om['effect']
    print(f"\nAre countries different in per-school all-three-window coverage? "
          f"Kruskal-Wallis H={_om['statistic']:.0f}, p={_om['p_value']:.1e}, "
          f"epsilon^2={_eff:.3f} "
          f"({'large' if _eff > 0.14 else 'moderate' if _eff > 0.06 else 'small'} separation; "
          f"n={len(_elig):,} schools with >=20 school-days)")
    _med = pd.Series(_om['medians']).sort_values(ascending=False)
    print(f"        per-school median all-three rate: best {_med.index[0]} {_med.iloc[0]:.1f}% · "
          f"worst {_med.index[-1]} {_med.iloc[-1]:.1f}% · fleet median school {_elig['pct_all3'].median():.1f}%")

    fig, axes = plt.subplots(1, 3, figsize=(17, max(4, 0.32 * len(slots))),
                             gridspec_kw={'width_ratios': [1, 1, 1.2]})
    _p = slots.sort_values('pct_days_all3')
    _y = np.arange(len(_p))
    axes[0].barh(_y - .27, _p['pct_days_morning'], height=.26, color=GIGA_PRIMARY[200],
                 label=f'morning {_s1}-{_s2}')
    axes[0].barh(_y, _p['pct_days_midday'], height=.26, color=GIGA_PRIMARY[500],
                 label=f'midday {_s2}-{_s3}')
    axes[0].barh(_y + .27, _p['pct_days_afternoon'], height=.26, color=GIGA_PRIMARY[800],
                 label=f'afternoon {_s3}-{_s4}')
    axes[0].set_yticks(_y); axes[0].set_yticklabels(_p.index, fontsize=8)
    axes[0].set_xlabel('% of school-days the window is hit')
    axes[0].set_title('Which window gets measured'); axes[0].legend(fontsize=8)
    axes[1].barh(_y, _p['pct_days_all3'], color=GIGA_GOOD, label='all three windows')
    axes[1].barh(_y, _p['pct_days_1and3_not2'], left=_p['pct_days_all3'], color=GIGA_MODERATE,
                 label='morning + afternoon, no midday')
    axes[1].set_yticks(_y); axes[1].set_yticklabels([]); axes[1].set_xlabel('% of school-days')
    axes[1].set_title('Full-day coverage vs the midday gap'); axes[1].legend(fontsize=8)

    _h = (slot_hours[slot_hours['iso3_code'].isin(slots.index)]
          .pivot_table(index='iso3_code', columns='hr', values='n', fill_value=0)
          .reindex(_p.index))
    _h = 100 * _h.div(_h.sum(axis=1), axis=0)
    sns.heatmap(_h, ax=axes[2], cmap='rocket_r', cbar_kws={'label': '% of measurements'},
                linewidths=.3, linecolor='white')
    for _b in (_s1, _s2, _s3, _s4):
        if _b in list(_h.columns):
            axes[2].axvline(list(_h.columns).index(_b), color=GIGA_BLUE, lw=1.2)
    axes[2].set_xlabel('local hour'); axes[2].set_ylabel('')
    axes[2].set_title('When in the day measurements land (lines = window edges)')
    axes[2].tick_params(labelsize=8)
    plt.tight_layout(); plt.show()

    print("Reading it: the three windows are what the app tries to do, so the gap between a window's")
    print("hit-rate and 100% is the schedule not completing — device asleep, browser closed, or the")
    print("connection down through all 3 retries. 'Morning + afternoon, no midday' is the diagnostic")
    print("shape: the device was demonstrably awake at both ends of the day, so the midday miss is a")
    print("lost window rather than a school that was simply offline.")
except Exception as _e:
    slots = pd.DataFrame(); slots_school = pd.DataFrame(); slot_hours = pd.DataFrame()
    print(f"(slot coverage needs Trino: {_e})")

---
## Part 3f — Ping coverage against the 48-a-day target

Separate from the speed test, the app runs a **connection check every 15 minutes
between 08:00 and 20:00 local** — 48 checks a day from a device that stays awake.
That fixed denominator makes the ping a cleaner presence signal than the
measurement: a device either kept the beat or it did not.

This asks how far short of 48 each fleet falls, and whether the shortfall is a
property of the **device** (some machines are reliable, others are not) or of the
**day** (the same machine is good one day and gone the next).

In [ ]:
# =============================================================================
# PING YIELD — how close does a device get to the 48 connection checks a day?
# =============================================================================
# The connection check is NOT a speed test. While the app is open it pings
# Cloudflare every 15 minutes between 08:00 and 20:00 local, so a device that is
# awake all day logs 48 checks (`expected_pings_per_day` = 48 in the upstream
# aggregate, confirmed against the raw 15-minute cadence). Source and caveats:
#  · `gigameter_production_db.public.connectivity_ping_checks` is the raw table.
#    `all_ping_daily` carries the same numbers but only starts 2026-05-16 and
#    inherits the timezone gap below, so the raw table is used instead.
#  · `timestamp` is UTC with NO local column, so the day and the hour are stamped
#    here with the country's own zone — the ping-side twin of the _LTS repair.
#    Zones come from `default.country_timezones` (61 countries), with
#    resolve_country() covering the handful it misses.
#  · soft-deleted rows are excluded (`is_deleted`) and future-dated corruption is
#    capped at current_timestamp — the raw table runs out to the year 3026.
#  · the grain is the DEVICE (`browser_id`), not the school: 48 is what one
#    device owes, and a school with two devices can log 96.
#  · counts are DISTINCT quarter-hours, not rows: 1.4% of quarters carry a repeat
#    ping, which would otherwise push a device past its own ceiling.
PING_SINCE          = '2026-02-01'   # V3 rollout; earlier rows are a trickle, mostly soft-deleted
PING_EXPECTED       = 48             # 4 per hour x 12 hours, per the scheduler doc
PING_COMPLETE       = 44             # >= this many quarter-hours = a "complete" day (>=90%)
PING_MIN_SCHOOLS    = 20             # skip countries with fewer pinging schools
PING_DEV_MIN_DAYS   = 20             # a device needs this many ping-days to be judged

# --- timezone map for the ping side, keyed on the production `country.code` ---
_cur = get_trino_cursor()
def _pq(sql):
    _cur.execute(sql)
    return pd.DataFrame(_cur.fetchall(), columns=[c[0] for c in _cur.description])

_tzl = _pq("SELECT iso2, timezone FROM delta_lake.default.country_timezones")
_pc  = _pq(f"""SELECT DISTINCT c.code AS iso2, c.iso3_format AS iso3
               FROM gigameter_production_db.public.connectivity_ping_checks p
               JOIN gigameter_production_db.public.school s ON p.giga_id_school = s.giga_id_school
               JOIN gigameter_production_db.public.country c ON s.country_id = c.id
               WHERE NOT p.is_deleted AND p.timestamp >= timestamp '{PING_SINCE}'""")
_zones = dict(zip(_tzl['iso2'], _tzl['timezone']))
PING_TZ, PING_TZ_MISSING = {}, []
for _r in _pc.itertuples():
    if _r.iso3 in (EXCLUDE_COUNTRIES or []) or (COUNTRIES and _r.iso3 not in COUNTRIES):
        continue
    _z = _zones.get(_r.iso2)
    if not _z:
        try:
            _z = resolve_country(_r.iso3)['timezone']
        except Exception:
            _z = None
    (PING_TZ.setdefault(_r.iso2, _z) if _z else PING_TZ_MISSING.append(_r.iso3))
_extra = sorted(k for k in PING_TZ if k not in _zones)
print(f"ping timezones: {len(PING_TZ)} countries "
      f"({'+' + ', '.join(_extra) + ' from resolve_country' if _extra else 'all from country_timezones'})"
      + (f" · dropped, no zone: {', '.join(PING_TZ_MISSING)}" if PING_TZ_MISSING else ""))

_ptz_cases = "\n              ".join(
    f"WHEN '{k}' THEN CAST(p.timestamp AT TIME ZONE '{v}' AS timestamp)" for k, v in sorted(PING_TZ.items()))
_PLT = f"(CASE c.code\n              {_ptz_cases}\n           END)"

# --- shared ping CTE: quarter-hours pinged per device-day, split by slot -------
PING_CTE = f"""
WITH pg AS (
    SELECT c.iso3_format AS iso3_code, p.giga_id_school AS school_id_giga,
           p.browser_id AS device_id, {_PLT} AS lts, p.is_connected
    FROM gigameter_production_db.public.connectivity_ping_checks p
    JOIN gigameter_production_db.public.school s ON p.giga_id_school = s.giga_id_school
    JOIN gigameter_production_db.public.country c ON s.country_id = c.id
    WHERE NOT p.is_deleted
      AND p.timestamp >= timestamp '{PING_SINCE}' AND p.timestamp <= current_timestamp
      AND c.code IN ('{"', '".join(sorted(PING_TZ))}')
),
pq AS (                       -- one row per device x day x quarter-hour actually pinged
    SELECT iso3_code, school_id_giga, device_id, CAST(lts AS date) AS d,
           hour(lts) * 4 + minute(lts) / 15 AS qtr,
           bool_or(is_connected) AS ok    -- the quarter counts as up if any ping in it answered
    FROM pg GROUP BY 1, 2, 3, 4, 5
),
pday AS (                     -- 08:00-20:00 is qtr 32-79; the three slots are 16 quarters each
    SELECT iso3_code, school_id_giga, device_id, d,
           count_if(qtr >= 32 AND qtr < 80) AS q_day,
           count_if(qtr >= 32 AND qtr < 48) AS q1,
           count_if(qtr >= 48 AND qtr < 64) AS q2,
           count_if(qtr >= 64 AND qtr < 80) AS q3,
           count_if(qtr >= 32 AND qtr < 80 AND ok) AS q_day_ok,
           count_if(qtr >= 32 AND qtr < 48 AND ok) AS q1_ok,
           count_if(qtr >= 48 AND qtr < 64 AND ok) AS q2_ok,
           count_if(qtr >= 64 AND qtr < 80 AND ok) AS q3_ok,
           count(*) AS q_all
    FROM pq GROUP BY 1, 2, 3, 4
)
"""

Q_PING_COUNTRY = PING_CTE + f"""
SELECT iso3_code, count(DISTINCT school_id_giga) AS schools, count(DISTINCT device_id) AS devices,
       count(*) AS device_days,
       approx_percentile(CAST(q_day AS double), 0.5) AS median_pings,
       avg(CAST(q_day AS double)) AS mean_pings,
       100.0 * avg(IF(q_day >= {PING_COMPLETE}, 1e0, 0e0)) AS pct_days_complete,
       100.0 * avg(IF(q_day >= 24, 1e0, 0e0))              AS pct_days_half,
       100.0 * avg(IF(q_day <= 4,  1e0, 0e0))              AS pct_days_token,
       avg(CAST(q1 AS double)) AS mean_morning,
       avg(CAST(q2 AS double)) AS mean_midday,
       avg(CAST(q3 AS double)) AS mean_afternoon,
       min(d) AS ping_from, max(d) AS ping_to
FROM pday WHERE day_of_week(d) <= 5 AND q_day > 0
GROUP BY 1 HAVING count(DISTINCT school_id_giga) >= {PING_MIN_SCHOOLS}
ORDER BY 7 DESC
"""

Q_PING_HIST = PING_CTE + """
SELECT iso3_code, q_day, count(*) AS n
FROM pday WHERE day_of_week(d) <= 5 AND q_day > 0 GROUP BY 1, 2
"""

Q_PING_DEVICE = PING_CTE + f"""
SELECT iso3_code, device_id, count(*) AS days,
       avg(CAST(q_day AS double)) AS mean_pings, stddev(CAST(q_day AS double)) AS sd_pings,
       100.0 * avg(IF(q_day >= {PING_COMPLETE}, 1e0, 0e0)) AS pct_complete
FROM pday WHERE day_of_week(d) <= 5 AND q_day > 0
GROUP BY 1, 2 HAVING count(*) >= {PING_DEV_MIN_DAYS}
"""

# --- the same day at SCHOOL grain, split into its two factors ----------------
# 48 = 12 hours x 4 checks. A school-day can therefore fall short two ways: the
# device covers fewer HOURS, or it covers an hour thinly. Splitting them says
# which one to fix, and the school (not the device) is the grain a country team
# acts on — devices are deduplicated to distinct quarter-hours so a two-device
# school cannot exceed 48.
_SCHOOL_CTE = """,
sq AS (SELECT iso3_code, school_id_giga, d, qtr, bool_or(ok) AS ok
       FROM pq GROUP BY 1, 2, 3, 4),
sday AS (
    SELECT iso3_code, school_id_giga, d,
           count_if(qtr >= 32 AND qtr < 80) AS q_day,
           count_if(qtr >= 32 AND qtr < 80 AND ok) AS q_day_ok,
           count(DISTINCT IF(qtr >= 32 AND qtr < 80, qtr / 4)) AS hrs_covered
    FROM sq GROUP BY 1, 2, 3
)"""

Q_PING_SCHOOL = PING_CTE + _SCHOOL_CTE + f"""
SELECT iso3_code, count(DISTINCT school_id_giga) AS schools, count(*) AS school_days,
       approx_percentile(CAST(q_day AS double), 0.25) AS p25_checks,
       approx_percentile(CAST(q_day AS double), 0.50) AS median_checks,
       approx_percentile(CAST(q_day AS double), 0.75) AS p75_checks,
       approx_percentile(CAST(hrs_covered AS double), 0.50) AS median_hours,
       approx_percentile(CAST(q_day AS double) / hrs_covered, 0.50) AS median_per_hour,
       100.0 * avg(IF(q_day >= {PING_COMPLETE}, 1e0, 0e0)) AS pct_days_complete
FROM sday WHERE day_of_week(d) <= 5 AND q_day > 0
GROUP BY 1 HAVING count(DISTINCT school_id_giga) >= {PING_MIN_SCHOOLS}
ORDER BY 5 DESC
"""

Q_PING_FACTORS = PING_CTE + _SCHOOL_CTE + """
SELECT 'hours' AS factor, iso3_code, hrs_covered AS k, count(*) AS n
FROM sday WHERE day_of_week(d) <= 5 AND q_day > 0 GROUP BY 1, 2, 3
UNION ALL
SELECT 'checks_in_hour', iso3_code, k, count(*) FROM (
    SELECT iso3_code, school_id_giga, d, qtr / 4 AS hr, count(*) AS k
    FROM sq WHERE day_of_week(d) <= 5 AND qtr >= 32 AND qtr < 80
    GROUP BY 1, 2, 3, 4
) GROUP BY 1, 2, 3
"""

try:
    ping = _pq(Q_PING_COUNTRY).set_index('iso3_code')
    ping_hist = _pq(Q_PING_HIST)
    ping_dev = _pq(Q_PING_DEVICE)
    ping_dev = ping_dev[ping_dev['iso3_code'].isin(ping.index)]
    ping_school = _pq(Q_PING_SCHOOL).set_index('iso3_code').reindex(ping.index)
    ping_factors = _pq(Q_PING_FACTORS)
    ping_factors = ping_factors[ping_factors['iso3_code'].isin(ping.index)]

    print(f"\nWeekdays only · {PING_SINCE} to {ping['ping_to'].max()} · denominator = device-days with "
          f"at least one ping, i.e. days the device was demonstrably awake at some point.")
    print(f"Target = {PING_EXPECTED} checks (08:00-20:00 local, every 15 min); "
          f"'complete' = {PING_COMPLETE}+ ({100 * PING_COMPLETE // PING_EXPECTED}% of the target).\n")
    _show = ['schools', 'devices', 'device_days', 'median_pings', 'mean_pings', 'pct_days_complete',
             'pct_days_half', 'pct_days_token', 'mean_morning', 'mean_midday', 'mean_afternoon']
    display(ping[_show].round(1).style
            .background_gradient(cmap='RdYlGn', subset=['pct_days_complete', 'median_pings'])
            .background_gradient(cmap='RdYlGn_r', subset=['pct_days_token'])
            .format('{:,.0f}', subset=['schools', 'devices', 'device_days'])
            .format('{:.1f}', subset=[c for c in _show if c not in
                                      ('schools', 'devices', 'device_days')]))

    _w = ping['device_days']
    _fl = {c: float((ping[c] * _w).sum() / _w.sum())
           for c in ['mean_pings', 'pct_days_complete', 'pct_days_half',
                     'pct_days_token', 'mean_morning', 'mean_midday', 'mean_afternoon']}
    print(f"FLEET ({int(ping['schools'].sum()):,} schools, {int(ping['devices'].sum()):,} devices, "
          f"{int(_w.sum()):,} device-days): mean {_fl['mean_pings']:.1f} of {PING_EXPECTED} checks "
          f"({100 * _fl['mean_pings'] / PING_EXPECTED:.0f}% of the target) · "
          f"complete on {_fl['pct_days_complete']:.1f}% of device-days · "
          f"at least half on {_fl['pct_days_half']:.1f}% · "
          f"4 or fewer on {_fl['pct_days_token']:.1f}%")
    print(f"        by slot, out of 16 quarter-hours each: morning {_fl['mean_morning']:.1f} · "
          f"midday {_fl['mean_midday']:.1f} · afternoon {_fl['mean_afternoon']:.1f}")
    print(f"        country spread in complete days: {ping['pct_days_complete'].idxmax()} "
          f"{ping['pct_days_complete'].max():.1f}% down to {ping['pct_days_complete'].idxmin()} "
          f"{ping['pct_days_complete'].min():.1f}%")

    # Is the difference between countries real, and how much of the variation is
    # WHICH device rather than which day? The device is the unit for both.
    _om = kruskal_omnibus(ping_dev, unit_col='device_id', group_col='iso3_code',
                          value_col='mean_pings', min_per_cell=1)
    _eff = _om['effect']
    print(f"\nDo countries differ in per-device mean yield? Kruskal-Wallis H={_om['statistic']:.0f}, "
          f"p={_om['p_value']:.1e}, epsilon^2={_eff:.3f} "
          f"({'large' if _eff > 0.14 else 'moderate' if _eff > 0.06 else 'small'} separation; "
          f"n={len(ping_dev):,} devices with >={PING_DEV_MIN_DAYS} ping-days)")
    _vb = ping_dev['mean_pings'].var()
    _vw = (ping_dev['sd_pings'] ** 2).mean()
    print(f"        variance split: between devices {_vb:.1f} vs within a device {_vw:.1f} "
          f"-> {_vb / (_vb + _vw):.0%} of the variation is WHICH device, "
          f"{_vw / (_vb + _vw):.0%} is which day for the same device")
    print(f"        devices never logging a complete day: {(ping_dev['pct_complete'] == 0).mean():.0%} · "
          f"complete on >=50% of their days: {(ping_dev['pct_complete'] >= 50).mean():.1%} · "
          f"on >=80%: {(ping_dev['pct_complete'] >= 80).mean():.1%}")

    # ---- the same shortfall at school grain, split into hours x checks-per-hour
    print(f"\nSCHOOL-DAY view — the grain a country team acts on, devices merged:")
    _s = ['schools', 'school_days', 'p25_checks', 'median_checks', 'p75_checks',
          'median_hours', 'median_per_hour', 'pct_days_complete']
    display(ping_school[_s].round(1).style
            .background_gradient(cmap='RdYlGn', subset=['median_checks', 'median_hours'])
            .format('{:,.0f}', subset=['schools', 'school_days'])
            .format('{:.1f}', subset=[c for c in _s if c not in ('schools', 'school_days')]))

    _ws = ping_school['school_days']
    _fs = {c: float((ping_school[c] * _ws).sum() / _ws.sum())
           for c in ['p25_checks', 'median_checks', 'p75_checks', 'median_hours', 'median_per_hour']}
    _hrs = (ping_factors[ping_factors['factor'] == 'hours']
            .groupby('k')['n'].sum().reindex(range(0, 13), fill_value=0))
    _cih = (ping_factors[ping_factors['factor'] == 'checks_in_hour']
            .groupby('k')['n'].sum().reindex(range(1, 5), fill_value=0))
    print(f"FLEET, per school-day (expected {PING_EXPECTED}): median {_fs['median_checks']:.0f} checks · "
          f"p25 {_fs['p25_checks']:.0f} · p75 {_fs['p75_checks']:.0f}")
    print(f"        hours covered (of 12): median {_fs['median_hours']:.0f} · "
          f"checks per COVERED hour (of 4): median {_fs['median_per_hour']:.1f}, "
          f"and {100 * _cih[4] / _cih.sum():.0f}% of covered hours are the full 4")
    print(f"        so the shortfall is HOURS, not intensity: an hour the device is present for is "
          f"essentially complete, and the median school-day covers "
          f"{_fs['median_hours']:.0f} of the 12 hours the app plans for.")
    print(f"        (medians of the two factors are separate order statistics — they do not multiply "
          f"back to the median day.)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    axes[0].bar(_hrs.index, 100 * _hrs / _hrs.sum(), color=GIGA_PRIMARY[500], width=.85)
    axes[0].axvline(12, color=GIGA_BAD, lw=1.4)
    axes[0].set_xlabel('hours of the 08:00-20:00 window with any check (of 12)')
    axes[0].set_ylabel('% of school-days'); axes[0].set_title('Factor 1 — hours covered')
    axes[1].bar(_cih.index, 100 * _cih / _cih.sum(), color=GIGA_GOOD, width=.6)
    axes[1].set_xticks([1, 2, 3, 4]); axes[1].set_xlabel('checks logged in a covered hour (of 4)')
    axes[1].set_ylabel('% of covered school-hours'); axes[1].set_title('Factor 2 — checks per covered hour')
    plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={'width_ratios': [1.2, 1, 1]})

    _h = ping_hist[ping_hist['iso3_code'].isin(ping.index)].groupby('q_day')['n'].sum()
    _h = _h.reindex(range(1, PING_EXPECTED + 1), fill_value=0)
    axes[0].bar(_h.index, 100 * _h / _h.sum(), color=GIGA_PRIMARY[500], width=.9)
    axes[0].axvline(PING_EXPECTED, color=GIGA_BAD, lw=1.4)
    axes[0].axvline(PING_COMPLETE, color=GIGA_MODERATE, lw=1.2, ls='--')
    axes[0].annotate(f'target {PING_EXPECTED}', xy=(PING_EXPECTED, axes[0].get_ylim()[1] * .92),
                     ha='right', fontsize=8, color=GIGA_BAD)
    axes[0].set_xlabel(f'connection checks logged in the day (of {PING_EXPECTED})')
    axes[0].set_ylabel('% of device-days')
    axes[0].set_title('Ping yield per device-day')

    _p = ping.sort_values('median_pings')
    _y = np.arange(len(_p))
    axes[1].barh(_y - .22, _p['mean_morning'], height=.22, color=GIGA_PRIMARY[200], label='morning')
    axes[1].barh(_y, _p['mean_midday'], height=.22, color=GIGA_PRIMARY[500], label='midday')
    axes[1].barh(_y + .22, _p['mean_afternoon'], height=.22, color=GIGA_PRIMARY[800], label='afternoon')
    axes[1].axvline(16, color=GIGA_BAD, lw=1.2)
    axes[1].set_yticks(_y); axes[1].set_yticklabels(_p.index, fontsize=8)
    axes[1].set_xlabel('mean quarter-hours pinged per slot (target 16)')
    axes[1].set_title('Where in the day the checks stop'); axes[1].legend(fontsize=8)

    _d = ping_dev.sort_values('mean_pings')
    axes[2].hist(_d['mean_pings'], bins=48, range=(0, PING_EXPECTED), color=GIGA_PRIMARY[600])
    axes[2].axvline(PING_EXPECTED, color=GIGA_BAD, lw=1.4)
    axes[2].axvline(_d['mean_pings'].median(), color=GIGA_BLUE, lw=1.2, ls='--')
    axes[2].set_xlabel(f"device's own mean yield across its days (of {PING_EXPECTED})")
    axes[2].set_ylabel('devices')
    axes[2].set_title(f'Are some devices reliable? ({len(_d):,} devices, >={PING_DEV_MIN_DAYS} days)')
    plt.tight_layout(); plt.show()

    print("Reading it: the target is not nearly met anywhere, so the interesting quantity is not")
    print("'who reaches 48' but how far short each fleet falls and whether the shortfall is a")
    print("property of the device or of the day. The device histogram is unimodal and short of the")
    print("target rather than split into reliable and unreliable populations — but a third of the")
    print("variance still sits between devices, so a device's yield is partly a stable trait.")
except Exception as _e:
    ping = pd.DataFrame(); ping_hist = pd.DataFrame(); ping_dev = pd.DataFrame()
    ping_school = pd.DataFrame(); ping_factors = pd.DataFrame()
    print(f"(ping yield needs Trino: {_e})")

---
## Part 3g — When the checks stop

Two silences, measured separately. **Within the day**: the hour at which a
device's checks stop is the hour the machine slept or the browser closed.
**Across the calendar**: a country-wide collapse in pinging devices is a holiday
or an outage — told apart by whether it repeats and whether the low season in
Part 2 moves with it.

In [ ]:
# =============================================================================
# WHEN THE CHECKS STOP — through the day, and across the calendar
# =============================================================================
# Two different silences. Within a day: the ping is a heartbeat, so the hour at
# which a device's checks stop is the hour the browser was closed or the machine
# slept. Across the calendar: a country-wide collapse in pinging devices is a
# holiday or an outage, and the two are told apart by whether it repeats and
# whether the speed-test season (Part 2) moves with it.
PING_QUIET_FRAC = 0.4     # a day is "quiet" below this share of the country's trailing median
PING_QUIET_RUN  = 3       # report runs of this many consecutive quiet days or more
_PING_TRAIL     = 28      # days in the trailing median

Q_PING_HOUR = PING_CTE + """
, ph AS (
    SELECT iso3_code, device_id, d, qtr / 4 AS hr, count(*) AS qs
    FROM pq GROUP BY 1, 2, 3, 4
)
SELECT iso3_code, hr, count(*) AS device_day_hours, avg(CAST(qs AS double)) AS mean_q_in_hour
FROM ph WHERE day_of_week(d) <= 5 GROUP BY 1, 2
"""

Q_PING_CAL = PING_CTE + f"""
SELECT iso3_code, d, count(DISTINCT device_id) AS devices,
       avg(CAST(q_day AS double)) AS mean_pings,
       100.0 * avg(IF(q_day >= {PING_COMPLETE}, 1e0, 0e0)) AS pct_complete
FROM pday WHERE q_day > 0 GROUP BY 1, 2
"""

try:
    ping_hour = _pq(Q_PING_HOUR)
    ping_cal = _pq(Q_PING_CAL)
    ping_hour = ping_hour[ping_hour['iso3_code'].isin(ping.index)]
    ping_cal = ping_cal[ping_cal['iso3_code'].isin(ping.index)]
    ping_cal['d'] = pd.to_datetime(ping_cal['d'])

    # ---- within the day: how many devices are still alive, and how full the hour is
    _den = ping['device_days']
    _hr = (ping_hour.pivot_table(index='iso3_code', columns='hr', values='device_day_hours',
                                 fill_value=0).reindex(ping.index))
    _alive = 100 * _hr.div(_den, axis=0)                    # % of device-days with a ping that hour
    _yield = (ping_hour.pivot_table(index='iso3_code', columns='hr', values='mean_q_in_hour',
                                    fill_value=0).reindex(ping.index))
    _fa = 100 * _hr.sum() / _den.sum()
    _fy = (_hr * _yield).sum() / _hr.sum()                  # device-day-hour weighted

    _day = [h for h in range(8, 20) if h in _fa.index]
    print("Within the day (weekdays, fleet), out of 4 possible checks an hour:")
    print("   hour  " + "  ".join(f"{h:>4d}" for h in _day))
    print("  alive% " + "  ".join(f"{_fa[h]:4.0f}" for h in _day))
    print("  checks " + "  ".join(f"{_fy[h]:4.1f}" for h in _day))
    _first, _last = _day[0], _day[-1]
    _peak = int(_fa[_day].idxmax())
    print(f"\nAttrition, not throttling: when a device is pinging in an hour at all it logs "
          f"{_fy[_day].mean():.1f} of 4 checks on average — the hour it is present for is close to "
          f"full. What moves is how many devices are there at all: {_fa[_first]:.0f}% of device-days "
          f"at {_first:02d}:00, rising to {_fa[_peak]:.0f}% at {_peak:02d}:00 as machines are switched "
          f"on, then falling away to {_fa[_last]:.0f}% by {_last:02d}:00. The afternoon is not a "
          f"connectivity problem — it is an empty-room problem.")
    _outside = 100 * _hr[[h for h in _hr.columns if h < 8 or h >= 20]].sum().sum() / _hr.sum().sum()
    print(f"Checks logged outside the 08:00-20:00 window: {_outside:.1f}% of all quarter-hours. The "
          f"app should not ping there at all, so this is the residue of clock skew and any timezone "
          f"still mis-stamped — small enough not to disturb the window totals.")

    # ---- across the calendar: country-wide quiet stretches
    _rows = []
    for _iso, _g in ping_cal.groupby('iso3_code'):
        _g = _g.sort_values('d').set_index('d').asfreq('D')
        _dev = _g['devices'].fillna(0)
        _base = _dev.rolling(_PING_TRAIL, min_periods=7).median().shift(1)
        _quiet = (_dev < PING_QUIET_FRAC * _base) & _base.notna()
        _grp = (_quiet != _quiet.shift()).cumsum()
        for _, _run in _quiet[_quiet].groupby(_grp[_quiet]):
            if len(_run) >= PING_QUIET_RUN:
                _rows.append({'iso3_code': _iso, 'from': _run.index[0].date(), 'to': _run.index[-1].date(),
                              'days': len(_run),
                              'weekdays': int((_run.index.dayofweek < 5).sum()),
                              'devices_in_run': int(_dev.loc[_run.index].mean()),
                              'devices_before': int(_base.loc[_run.index[0]])})
    ping_quiet = pd.DataFrame(_rows).sort_values(['days', 'devices_before'], ascending=False)
    print(f"\nCountry-wide quiet stretches (>={PING_QUIET_RUN} consecutive days below "
          f"{PING_QUIET_FRAC:.0%} of the trailing {_PING_TRAIL}-day median device count):")
    if len(ping_quiet):
        display(ping_quiet.head(20).style.hide(axis='index')
                .format({'devices_in_run': '{:,.0f}', 'devices_before': '{:,.0f}'}))
        print(f"{len(ping_quiet)} stretches across {ping_quiet['iso3_code'].nunique()} countries. "
              "Weekday-heavy runs in term time are the ones worth chasing; a run that lands on the "
              "country's low season in Part 2 is the school year, not a broken fleet.")
    else:
        print("  none — no country loses its fleet for three days running.")

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={'width_ratios': [1, 1.2, 1.4]})

    axes[0].plot(_day, [_fa[h] for h in _day], marker='o', color=GIGA_PRIMARY[600],
                 label='% of device-days still pinging')
    axes[0].set_ylim(0, 105); axes[0].set_xlabel('local hour'); axes[0].set_ylabel('%')
    _ax2 = axes[0].twinx()
    _ax2.plot(_day, [_fy[h] for h in _day], marker='s', ms=4, color=GIGA_MODERATE, ls='--',
              label='checks logged in that hour (of 4)')
    _ax2.set_ylim(0, 4.2); _ax2.set_ylabel('checks per alive hour')
    axes[0].set_title('The day drains devices, not checks')
    _l1, _b1 = axes[0].get_legend_handles_labels(); _l2, _b2 = _ax2.get_legend_handles_labels()
    axes[0].legend(_l1 + _l2, _b1 + _b2, fontsize=8, loc='lower left')

    sns.heatmap(_alive[_day].reindex(ping.sort_values('median_pings').index), ax=axes[1],
                cmap='rocket_r', vmin=0, vmax=100, linewidths=.3, linecolor='white',
                cbar_kws={'label': '% of device-days pinging'})
    axes[1].set_xlabel('local hour'); axes[1].set_ylabel('')
    axes[1].set_title('How far into the day each fleet lasts'); axes[1].tick_params(labelsize=8)

    _f = (ping_cal.groupby('d').apply(
        lambda g: pd.Series({'devices': g['devices'].sum(),
                             'mean_pings': np.average(g['mean_pings'], weights=g['devices'])}),
        include_groups=False).sort_index())
    axes[2].fill_between(_f.index, _f['devices'], color=GIGA_PRIMARY[200], label='devices pinging')
    axes[2].set_ylabel('devices pinging that day'); axes[2].set_xlabel('')
    _ax3 = axes[2].twinx()
    _ax3.plot(_f.index, _f['mean_pings'].rolling(7, center=True).mean(), color=GIGA_BAD, lw=1.4,
              label='mean checks per device-day (7d)')
    _ax3.set_ylim(0, PING_EXPECTED); _ax3.set_ylabel(f'checks per device-day (of {PING_EXPECTED})')
    axes[2].set_title('Fleet size and yield over the ping window')
    _l1, _b1 = axes[2].get_legend_handles_labels(); _l2, _b2 = _ax3.get_legend_handles_labels()
    axes[2].legend(_l1 + _l2, _b1 + _b2, fontsize=8, loc='upper right')
    axes[2].tick_params(axis='x', labelsize=8, rotation=30)
    plt.tight_layout(); plt.show()

    print("Reading it: the two panels separate a device that never woke from one that woke and then")
    print("went away. Yield per alive hour barely moves across the day, so the shortfall against 48 is")
    print("almost entirely the machine being off — which is the same explanation Part 3e needed for")
    print("the missing afternoon speed test, now measured directly rather than inferred.")
except Exception as _e:
    ping_hour = pd.DataFrame(); ping_cal = pd.DataFrame(); ping_quiet = pd.DataFrame()
    print(f"(ping timing needs Trino: {_e})")

---
## Part 3h — The shape of the ping day

A row-per-school heatmap of hourly ping shares shows the pattern but cannot be
compared across countries — 15,000 rows is not a finding. The shape that repeats
is always the same one with the **right edge in a different place**, so each
school is reduced to a single ordinal coordinate: the hour by which it has logged
80% of its checks, binned into four day shapes.

That is what makes the pattern portable: a country becomes a **mix of four
shapes**, and a fleet that is mostly *morning* is structurally unable to fill the
midday and afternoon scheduling windows however well the app behaves.

In [ ]:
# =============================================================================
# DAY SHAPES — every school has an hourly profile; reduce it to one label
# =============================================================================
# A row-per-school heatmap of hourly ping shares shows the pattern but does not
# scale: 15,000 rows cannot be compared across countries. Each school's profile
# is summarised by ONE ordinal number — the hour by which it has logged 80% of
# its checks — and binned into four shapes. That keeps the thing the heatmap
# actually shows (how far into the day the beat lasts) and drops the rest.
#   · shares are computed per school over 08:00-20:00 on weekdays, pooled across
#     its days, so a school with more days does not get a wider profile
#   · schools need PING_SHAPE_MIN_CHECKS checks and PING_SHAPE_MIN_DAYS days,
#     otherwise the profile is one quiet week's noise
#   · "80% by 15:00" means the cumulative share crosses 80% during hour 14
PING_SHAPE_MIN_CHECKS = 200
PING_SHAPE_MIN_DAYS   = 20
PING_SHAPE_BINS  = [7, 12, 14, 16, 19]     # on the p80 hour
PING_SHAPE_LABELS = ['morning (80% by 13:00)', 'school day (by 15:00)',
                     'long day (by 17:00)', 'into the evening']

Q_PING_SHAPE = PING_CTE + _SCHOOL_CTE + """
SELECT iso3_code, school_id_giga, qtr / 4 AS hr,
       count(*) AS checks, count(DISTINCT d) AS days
FROM sq WHERE day_of_week(d) <= 5 AND qtr >= 32 AND qtr < 80
GROUP BY 1, 2, 3
"""

try:
    ping_shape_raw = _pq(Q_PING_SHAPE)
    ping_shape_raw = ping_shape_raw[ping_shape_raw['iso3_code'].isin(ping.index)]
    _r = ping_shape_raw.astype({'hr': int, 'checks': int, 'days': int})

    _tot = _r.groupby(['iso3_code', 'school_id_giga']).agg(total=('checks', 'sum'),
                                                           days=('days', 'max'))
    _ok = _tot[(_tot['total'] >= PING_SHAPE_MIN_CHECKS) & (_tot['days'] >= PING_SHAPE_MIN_DAYS)]
    _prof = (_r.pivot_table(index=['iso3_code', 'school_id_giga'], columns='hr',
                            values='checks', fill_value=0)
               .reindex(columns=range(8, 20), fill_value=0)
               .reindex(_ok.index))
    _sh = _prof.div(_prof.sum(axis=1), axis=0)
    _cum = _sh.cumsum(axis=1)

    ping_shapes = pd.DataFrame({
        'p50_hour': _cum.ge(0.50).idxmax(axis=1),
        'p80_hour': _cum.ge(0.80).idxmax(axis=1),
        'share_before_14': _sh.loc[:, 8:13].sum(axis=1),
        'share_after_16': _sh.loc[:, 16:19].sum(axis=1),
        'checks': _ok['total'], 'days': _ok['days'],
    }).reset_index()
    ping_shapes['shape'] = pd.cut(ping_shapes['p80_hour'], bins=PING_SHAPE_BINS,
                                  labels=PING_SHAPE_LABELS)

    _m = ping_shapes
    print(f"{len(_m):,} schools profiled (>={PING_SHAPE_MIN_CHECKS} checks and "
          f">={PING_SHAPE_MIN_DAYS} weekdays each), out of {len(_tot):,} with any ping.")
    print(f"Median school: half its checks are done by {int(_m['p50_hour'].median()) + 1:02d}:00, "
          f"80% by {int(_m['p80_hour'].median()) + 1:02d}:00, and "
          f"{100 * _m['share_before_14'].median():.0f}% of its checks land before 14:00.")
    print(f"        share of checks after 16:00: median {100 * _m['share_after_16'].median():.0f}%, "
          f"and only {(_m['share_after_16'] >= .2).mean():.1%} of schools put a fifth of their "
          f"checks there.\n")

    _mix = pd.crosstab(_m['iso3_code'], _m['shape'], normalize='index') * 100
    _mix = _mix.reindex(columns=PING_SHAPE_LABELS)
    _mix['schools'] = _m.groupby('iso3_code').size()
    _mix = _mix[_mix['schools'] >= PING_MIN_SCHOOLS].sort_values(PING_SHAPE_LABELS[0], ascending=False)
    ping_shape_mix = _mix
    display(_mix.round(1).style
            .background_gradient(cmap='Blues', subset=PING_SHAPE_LABELS, vmin=0, vmax=80)
            .format('{:.1f}', subset=PING_SHAPE_LABELS).format('{:,.0f}', subset=['schools'])
            .set_caption('% of a country\'s schools in each day shape'))

    _fleetmix = (_m['shape'].value_counts(normalize=True).reindex(PING_SHAPE_LABELS) * 100)
    print("FLEET mix: " + " · ".join(f"{l} {_fleetmix[l]:.0f}%" for l in PING_SHAPE_LABELS))
    for _l in PING_SHAPE_LABELS:
        _c = _mix[_mix[_l] == _mix[_l].max()].index[0]
        print(f"        most {_l:<24} {_c} ({_mix.loc[_c, _l]:.0f}% of its schools)")

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={'width_ratios': [1, 1, 1.3]})
    _hours = list(range(8, 20))

    _bar = _prof.sum(axis=0).reindex(_hours, fill_value=0)
    axes[0].bar(_hours, _bar.values, color=GIGA_PRIMARY[500], width=.85)
    axes[0].set_xlabel('local hour'); axes[0].set_ylabel('ping checks')
    axes[0].set_title('Ping checks by local hour — all schools')

    _cent = _sh.groupby(_m.set_index(['iso3_code', 'school_id_giga'])['shape'],
                        observed=False).mean().reindex(PING_SHAPE_LABELS)
    sns.heatmap(100 * _cent, ax=axes[1], cmap='Blues', linewidths=.4, linecolor='white',
                cbar_kws={'label': '% of the school\'s checks'})
    axes[1].set_xlabel('local hour'); axes[1].set_ylabel('')
    axes[1].set_title('What each shape looks like (mean profile)')
    axes[1].tick_params(labelsize=8)

    _p = _mix[PING_SHAPE_LABELS].iloc[::-1]
    _y = np.arange(len(_p)); _left = np.zeros(len(_p))
    for _l, _c in zip(PING_SHAPE_LABELS, [GIGA_PRIMARY[200], GIGA_PRIMARY[400],
                                          GIGA_PRIMARY[600], GIGA_PRIMARY[800]]):
        axes[2].barh(_y, _p[_l], left=_left, color=_c, label=_l, height=.75)
        _left = _left + _p[_l].values
    axes[2].set_yticks(_y); axes[2].set_yticklabels(_p.index, fontsize=8)
    axes[2].set_xlim(0, 100); axes[2].set_xlabel('% of schools')
    axes[2].set_title('Day shape mix by country'); axes[2].legend(fontsize=7, loc='lower right')
    plt.tight_layout(); plt.show()

    print("Reading it: the per-school heatmap repeats one shape with a moving right edge, so the")
    print("edge is the only coordinate worth keeping. Countries then separate cleanly — a fleet that")
    print("is mostly 'morning' cannot fill the midday and afternoon scheduling windows no matter how")
    print("reliable the app is, which is the constraint Part 3i has to be read against.")
except Exception as _e:
    ping_shapes = pd.DataFrame(); ping_shape_mix = pd.DataFrame(); ping_shape_raw = pd.DataFrame()
    print(f"(day shapes need Trino: {_e})")

---
## Part 3i — Ping against speed test: dead device or lost test?

Part 3e could not say why a scheduling window produced no measurement. The ping
supplies the missing witness: for each school-day and window, was the device
**demonstrably online with its link up** — and if it was, did the speed test still
land? A device that is on with a dead line is separated out: that miss belongs to
the ISP, not the scheduler.

This turns the time-of-day result from a description into a diagnosis, and it is
the number to hand a country team: attrition needs the school to keep the machine
on, a lost test with the device online needs the app fixed.

In [ ]:
# =============================================================================
# PING vs SPEED TEST — was the missed window a dead device or a lost test?
# =============================================================================
# Part 3e could not tell the two apart: a window with no measurement might be a
# machine that was off, or a machine that was on while the scheduled test failed.
# The ping is the missing witness. For each school-day and each 4-hour window the
# device is called ONLINE when it logged at least PING_ONLINE_Q of the window's 16
# quarter-hours, and the question becomes conditional: given the device was there,
# did the speed test land?
#   · ONLINE is further split by whether those checks ANSWERED: a device that is
#     on with a dead line cannot run a speed test either, and that is the ISP's
#     problem, not the scheduler's
#   · ping is device-grain and measurement is school-grain, so a school counts as
#     online in a window if ANY of its devices was — the generous reading, which
#     makes the conditional test-rate a LOWER bound on scheduler reliability
#   · both sides are keyed on the same local date, each repaired with its own
#     timezone map (_LTS for measurements, PING_TZ for pings)
#   · the denominator is school-days with at least one ping anywhere in the day,
#     inside the ping window only — so this is the V3 cohort, not the whole fleet
PING_ONLINE_Q = 12        # of 16 quarter-hours in a window; 12 = online for >= 45 min in each hour
PING_LINK_UP  = 0.8       # of those quarters, the share that must have ANSWERED for the link to
                          # count as up — a device that is on with a dead line cannot run a test,
                          # and blaming the scheduler for that would be wrong

Q_PING_VS_TEST = PING_CTE + f""",
pschool AS (
    SELECT iso3_code, school_id_giga, d,
           max(q1) AS pq1, max(q2) AS pq2, max(q3) AS pq3, max(q_day) AS pq_day,
           max(q1_ok) AS pq1_ok, max(q2_ok) AS pq2_ok, max(q3_ok) AS pq3_ok
    FROM pday GROUP BY 1, 2, 3
),
mslot AS (
    SELECT iso3_code, school_id_giga, CAST({_LTS} AS date) AS d,
           max(IF(hour({_LTS}) >=  8 AND hour({_LTS}) < 12, 1, 0)) AS m1,
           max(IF(hour({_LTS}) >= 12 AND hour({_LTS}) < 16, 1, 0)) AS m2,
           max(IF(hour({_LTS}) >= 16 AND hour({_LTS}) < 20, 1, 0)) AS m3
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL AND created_timestamp IS NOT NULL
      AND CAST({_LTS} AS date) >= DATE '{PING_SINCE}'
      {_TZ_DROP_SQL} {_scope_sql}
    GROUP BY 1, 2, 3
),
j AS (
    SELECT p.iso3_code, p.d, p.pq1, p.pq2, p.pq3, p.pq1_ok, p.pq2_ok, p.pq3_ok,
           COALESCE(m.m1, 0) AS m1, COALESCE(m.m2, 0) AS m2, COALESCE(m.m3, 0) AS m3
    FROM pschool p
    LEFT JOIN mslot m
           ON m.iso3_code = p.iso3_code AND m.school_id_giga = p.school_id_giga AND m.d = p.d
    WHERE day_of_week(p.d) <= 5 AND p.pq_day > 0
)
SELECT iso3_code, slot,
       CASE WHEN pq >= {PING_ONLINE_Q} AND pq_ok >= {PING_LINK_UP} * pq THEN 'online, link up'
            WHEN pq >= {PING_ONLINE_Q} THEN 'online, link down'
            WHEN pq > 0 THEN 'partial' ELSE 'absent' END AS ping_state,
       CAST(m AS bigint) AS tested,
       count(*) AS n
FROM j CROSS JOIN UNNEST(ARRAY[
        ROW('morning', pq1, pq1_ok, m1), ROW('midday', pq2, pq2_ok, m2),
        ROW('afternoon', pq3, pq3_ok, m3)
     ]) AS t(slot, pq, pq_ok, m)
GROUP BY 1, 2, 3, 4
"""

try:
    pvt = _pq(Q_PING_VS_TEST)
    pvt = pvt[pvt['iso3_code'].isin(ping.index)]
    _order = ['morning', 'midday', 'afternoon']
    _STATES = ['online, link up', 'online, link down', 'partial', 'absent']

    def _slot_table(df):
        """One row per slot: where the school-days sit, and the conditional test rate."""
        t = df.pivot_table(index='slot', columns=['ping_state', 'tested'], values='n',
                           aggfunc='sum', fill_value=0).reindex(_order)
        out = pd.DataFrame(index=t.index)
        _tot = t.sum(axis=1)
        for _st in _STATES:
            _n = t.get((_st, 0), 0) + t.get((_st, 1), 0)
            out[f'% days {_st}'] = 100 * _n / _tot
            out[f'test rate | {_st}'] = 100 * t.get((_st, 1), 0) / _n.replace(0, np.nan)
        out['% days tested'] = 100 * sum(t.get((s, 1), 0) for s in _STATES) / _tot
        out['school-days'] = _tot
        return out

    fleet_slots = _slot_table(pvt)
    print(f"School-days in the ping window with at least one ping, weekdays "
          f"({int(fleet_slots['school-days'].iloc[0]):,} per slot, {ping_dev['iso3_code'].nunique()} countries).")
    print(f"ONLINE = {PING_ONLINE_Q}+ of the window's 16 quarter-hours pinged.\n")
    display(fleet_slots.round(1).style
            .background_gradient(cmap='RdYlGn', subset=['test rate | online'])
            .format('{:,.0f}', subset=['school-days'])
            .format('{:.1f}', subset=[c for c in fleet_slots.columns if c != 'school-days']))

    for _s in _order:
        _r = fleet_slots.loc[_s]
        print(f"{_s:>9}: online with the link up on {_r['% days online, link up']:.0f}% of school-days "
              f"and a test lands on {_r['test rate | online, link up']:.0f}% of those · "
              f"online with the link down {_r['% days online, link down']:.1f}% · "
              f"absent on {_r['% days absent']:.0f}% · window measured overall "
              f"{_r['% days tested']:.0f}%")

    # Decompose the misses: of every window that produced no speed test, how much
    # is the machine not being there at all?
    _miss = pvt[pvt['tested'] == 0].pivot_table(index='slot', columns='ping_state', values='n',
                                                aggfunc='sum', fill_value=0).reindex(_order)
    _miss_pct = 100 * _miss.div(_miss.sum(axis=1), axis=0)
    print("\nOf the windows with NO speed test, the device was:")
    for _s in _order:
        print(f"{_s:>9}: absent {_miss_pct.loc[_s, 'absent']:.0f}% · "
              f"partly there {_miss_pct.loc[_s, 'partial']:.0f}% · "
              f"online but the link was down {_miss_pct.loc[_s, 'online, link down']:.1f}% · "
              f"online with a working link {_miss_pct.loc[_s, 'online, link up']:.0f}%  "
              f"<- only this last group is the scheduler losing a test")

    # Per country, the number that matters: the test rate with the device online.
    _cty = (pvt.groupby(['iso3_code', 'slot', 'ping_state'])
               .apply(lambda g: pd.Series({'n': g['n'].sum(),
                                           'tested': g.loc[g['tested'] == 1, 'n'].sum()}),
                      include_groups=False).reset_index())
    _on = _cty[_cty['ping_state'] == 'online, link up'].copy()
    _on['test_rate'] = 100 * _on['tested'] / _on['n']
    ping_vs_test = (_on.pivot_table(index='iso3_code', columns='slot', values='test_rate')
                       .reindex(columns=_order))
    ping_vs_test['school-days online (morning)'] = (
        _on[_on['slot'] == 'morning'].set_index('iso3_code')['n'])
    display(ping_vs_test.round(1).style
            .background_gradient(cmap='RdYlGn', subset=_order, vmin=40, vmax=100)
            .format('{:.1f}', subset=_order).format('{:,.0f}', subset=['school-days online (morning)'])
            .set_caption('Speed-test rate GIVEN the device was pinging through the window (%)'))

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    _x = np.arange(len(_order))
    axes[0].bar(_x - .2, fleet_slots['% days online, link up'], .38, color=GIGA_PRIMARY[300],
                label='device online, link up')
    axes[0].bar(_x + .2, fleet_slots['% days tested'], .38, color=GIGA_PRIMARY[700],
                label='speed test landed')
    axes[0].set_xticks(_x); axes[0].set_xticklabels(_order)
    axes[0].set_ylabel('% of school-days'); axes[0].legend(fontsize=8)
    axes[0].set_title('Presence and measurement, by window')

    _b = np.zeros(len(_order))
    for _st, _c, _lab in [('absent', GIGA_BAD, 'device absent'),
                          ('partial', GIGA_MODERATE, 'device partly there'),
                          ('online, link down', GIGA_BLUE, 'device online, link down'),
                          ('online, link up', GIGA_GOOD, 'device online — test lost')]:
        axes[1].bar(_x, _miss_pct[_st], .6, bottom=_b, color=_c, label=_lab)
        _b = _b + _miss_pct[_st].values
    axes[1].set_xticks(_x); axes[1].set_xticklabels(_order)
    axes[1].set_ylabel('% of windows with no speed test'); axes[1].legend(fontsize=8)
    axes[1].set_title('Why a window produced nothing')

    _p = ping_vs_test[_order].dropna(how='all').sort_values('morning')
    _y = np.arange(len(_p))
    for _i, (_s, _c) in enumerate(zip(_order, [GIGA_PRIMARY[200], GIGA_PRIMARY[500], GIGA_PRIMARY[800]])):
        axes[2].barh(_y + (_i - 1) * .27, _p[_s], height=.26, color=_c, label=_s)
    axes[2].set_yticks(_y); axes[2].set_yticklabels(_p.index, fontsize=8)
    axes[2].set_xlim(0, 100); axes[2].set_xlabel('% of online windows that produced a test')
    axes[2].set_title('Scheduler reliability with the device present'); axes[2].legend(fontsize=8)
    plt.tight_layout(); plt.show()

    print("Reading it: the answer to Part 3e is mostly 'the machine was not there'. With the device")
    print("demonstrably online the scheduled test lands the large majority of the time, and the")
    print("residual grows through the day — so there is a real, smaller scheduler loss on top of the")
    print("attrition, and the afternoon window is where both problems land at once.")
    print("The caveat is the 'absent' column: a test does land in some windows with no ping at all,")
    print("so the ping is not a complete witness — a second device on a build that does not ping, or")
    print("a ping path blocked while the test path works, both look like an absent device here.")
except Exception as _e:
    pvt = pd.DataFrame(); fleet_slots = pd.DataFrame(); ping_vs_test = pd.DataFrame()
    print(f"(ping vs speed test needs Trino: {_e})")

---
## Part 3j — Coverage x success: did it run, and did it answer?

Everything up to here counts checks that **ran**. `is_connected` is the second
dimension, and it belongs to a different owner, so the ping splits into three
ratios that multiply:

| | | owner |
|---|---|---|
| **coverage** | recorded / expected | the school's device — was it on? |
| **success** | responded / recorded | the school's ISP — was the link up? |
| **effective** | responded / expected | what the school actually gets |

Two schools with the same effective number can need opposite interventions, which
is the whole reason to keep the factors apart — and it is why every figure in
Parts 3f–3i, which counts only checks that ran, needs this one beside it.

In [ ]:
# =============================================================================
# COVERAGE x SUCCESS — did the check run, and did it find a connection?
# =============================================================================
# Everything up to here counts checks that RAN. `is_connected` is the second
# dimension and a different owner's problem, so the ping splits into three ratios
# that multiply:
#     COVERAGE  = recorded / expected    did the device run the check?   (the school's device)
#     SUCCESS   = responded / recorded   was the link up when it did?    (the school's ISP)
#     EFFECTIVE = responded / expected   = coverage x success            (what the school actually gets)
# Read per school over the days it pinged at all, so COVERAGE is "of the 48 owed
# on a day the device was alive" and not diluted by holidays. Two schools with the
# same effective number need opposite interventions, which is the whole point of
# keeping the factors apart.
PING_SUCCESS_OK   = 90     # % — a link at or above this is treated as healthy
PING_SCHOOL_MIN_D = 20     # school needs this many ping-days to be plotted

# Coverage is counted on deduplicated quarter-hours (a school cannot run more
# than 48 of them), but SUCCESS is counted on the raw ping rows — "responded /
# recorded" is a property of the checks themselves, and crediting a whole quarter
# to one answered ping out of two would flatter it.
Q_PING_CS = PING_CTE + _SCHOOL_CTE + f""",
praw AS (
    SELECT iso3_code, school_id_giga, CAST(lts AS date) AS d,
           count(*) AS pings, count_if(is_connected) AS pings_ok
    FROM pg WHERE hour(lts) >= 8 AND hour(lts) < 20 GROUP BY 1, 2, 3
),
agg AS (
    SELECT s.iso3_code, s.school_id_giga, count(*) AS days,
           sum(s.q_day) AS recorded, sum(s.q_day_ok) AS responded_q,
           sum(r.pings) AS pings, sum(r.pings_ok) AS pings_ok,
           {PING_EXPECTED} * count(*) AS expected
    FROM sday s JOIN praw r
      ON r.iso3_code = s.iso3_code AND r.school_id_giga = s.school_id_giga AND r.d = s.d
    WHERE day_of_week(s.d) <= 5 AND s.q_day > 0
    GROUP BY 1, 2 HAVING count(*) >= {PING_SCHOOL_MIN_D}
)
SELECT * FROM agg
"""

try:
    cs = _pq(Q_PING_CS)
    cs = cs[cs['iso3_code'].isin(ping.index)].copy()
    for _c in ['recorded', 'responded_q', 'expected', 'days', 'pings', 'pings_ok']:
        cs[_c] = cs[_c].astype(float)
    cs['coverage']  = 100 * cs['recorded'] / cs['expected']
    cs['success']   = 100 * cs['pings_ok'] / cs['pings']
    cs['effective'] = cs['coverage'] * cs['success'] / 100
    ping_cs = cs

    _q = [10, 50, 90, 100]
    print(f"Per school ({len(cs):,} schools with >={PING_SCHOOL_MIN_D} weekday ping-days), "
          f"against {PING_EXPECTED} expected checks on each of those days:")
    for _lab, _col, _gloss in [('COVERAGE ', 'coverage',  'recorded / expected  (did the device run?)'),
                               ('SUCCESS  ', 'success',   'responded / recorded (was the connection up?)'),
                               ('EFFECTIVE', 'effective', 'responded / expected (both)')]:
        _p = np.percentile(cs[_col], _q)
        print(f"{_lab} {_gloss}\n"
              f"   p10 {_p[0]:.0f}%  ·  median {_p[1]:.0f}%  ·  p90 {_p[2]:.0f}%  ·  max {_p[3]:.0f}%")
    print(f"\nThe two factors are near-independent (Spearman rho = "
          f"{cs['coverage'].corr(cs['success'], method='spearman'):+.2f}): a school that keeps its "
          f"device on is not thereby a school with a good link. Coverage is the binding constraint — "
          f"its median is {cs['coverage'].median():.0f}% while success sits at "
          f"{cs['success'].median():.0f}%, so effective coverage is lost mostly to devices, not lines.")

    # Quadrants: which lever does each school need?
    _covmed = cs['coverage'].median()
    cs['quadrant'] = np.select(
        [(cs['coverage'] >= _covmed) & (cs['success'] >= PING_SUCCESS_OK),
         (cs['coverage'] >= _covmed) & (cs['success'] < PING_SUCCESS_OK),
         (cs['coverage'] < _covmed) & (cs['success'] >= PING_SUCCESS_OK)],
        ['device on, link good', 'device on, link flaky', 'device off, link good'],
        default='device off, link flaky')
    _QORD = ['device on, link good', 'device on, link flaky',
             'device off, link good', 'device off, link flaky']
    print(f"\nSplit at coverage {_covmed:.0f}% (the fleet median school) and success "
          f"{PING_SUCCESS_OK}%:")
    for _qd in _QORD:
        _n = (cs['quadrant'] == _qd).sum()
        print(f"   {_qd:<24} {_n:>6,} schools ({100 * _n / len(cs):4.1f}%)")

    _cty = cs.groupby('iso3_code').agg(schools=('school_id_giga', 'size'),
                                       coverage=('coverage', 'median'),
                                       success=('success', 'median'),
                                       effective=('effective', 'median'))
    _mix = (pd.crosstab(cs['iso3_code'], cs['quadrant'], normalize='index') * 100
            ).reindex(columns=_QORD, fill_value=0)
    _cty = _cty.join(_mix).sort_values('effective', ascending=False)
    ping_cs_country = _cty
    display(_cty.round(1).style
            .background_gradient(cmap='RdYlGn', subset=['coverage', 'success', 'effective'])
            .format('{:,.0f}', subset=['schools'])
            .format('{:.1f}', subset=[c for c in _cty.columns if c != 'schools'])
            .set_caption('Median school per country, and the quadrant mix (%)'))

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw={'width_ratios': [1.1, 1.1, 1]})

    axes[0].scatter(cs['coverage'], cs['success'], s=7, alpha=.18,
                    color=GIGA_PRIMARY[600], edgecolors='none')
    axes[0].axhline(PING_SUCCESS_OK, color=GIGA_GREY[400], ls='--', lw=1)
    axes[0].axvline(_covmed, color=GIGA_GREY[400], ls='--', lw=1)
    axes[0].set_xlim(0, 100); axes[0].set_ylim(0, 100)
    axes[0].set_xlabel('Coverage — % of expected ping checks that ran')
    axes[0].set_ylabel('Success — % of those that found a connection')
    axes[0].set_title(f'Every school ({len(cs):,})')

    axes[1].scatter(_cty['coverage'], _cty['success'], s=np.sqrt(_cty['schools']) * 9,
                    color=GIGA_PRIMARY[700], alpha=.75, edgecolors='white', linewidths=.8)
    for _i, _r in _cty.iterrows():
        axes[1].annotate(_i, (_r['coverage'], _r['success']), fontsize=7.5,
                         xytext=(4, 3), textcoords='offset points')
    axes[1].axhline(PING_SUCCESS_OK, color=GIGA_GREY[400], ls='--', lw=1)
    axes[1].axvline(_covmed, color=GIGA_GREY[400], ls='--', lw=1)
    axes[1].set_xlim(0, 60); axes[1].set_ylim(50, 100)
    axes[1].set_xlabel('Coverage — median school (%)'); axes[1].set_ylabel('Success — median school (%)')
    axes[1].set_title('Median school per country (area = schools)')

    _p = _mix.reindex(_cty.index)[_QORD].iloc[::-1]
    _y = np.arange(len(_p)); _left = np.zeros(len(_p))
    for _l, _c in zip(_QORD, [GIGA_GOOD, GIGA_MODERATE, GIGA_BLUE, GIGA_BAD]):
        axes[2].barh(_y, _p[_l], left=_left, color=_c, label=_l, height=.75)
        _left = _left + _p[_l].values
    axes[2].set_yticks(_y); axes[2].set_yticklabels(_p.index, fontsize=8)
    axes[2].set_xlim(0, 100); axes[2].set_xlabel('% of schools')
    axes[2].set_title('Which lever each fleet needs'); axes[2].legend(fontsize=7, loc='lower right')
    plt.tight_layout(); plt.show()

    print("Reading it: the cloud is wide on coverage and squashed against the top on success, which is")
    print("the shape of a device problem, not a connectivity problem. The schools worth routing to an")
    print("ISP conversation are the low-success tail — they are a minority, and they are invisible in")
    print("any metric that only counts checks that ran, including every number in Parts 3f-3i.")
except Exception as _e:
    ping_cs = pd.DataFrame(); ping_cs_country = pd.DataFrame()
    print(f"(coverage x success needs Trino: {_e})")

---
## Part 3k — Intensity: tests, not days

Part 1 counts the **days** a school measured. It never counts the **tests**: a
school measuring on twenty days may have run twenty readings or two hundred, and
those are different deployments telling the same story.

The app's ceiling is four automatic tests a day — one per scheduling window plus
the daily launch test — so anything above four is a person pressing the button,
and that is worth separating before any per-day average is quoted.

In [ ]:
# =============================================================================
# INTENSITY — Part 1 counts DAYS measured; this counts TESTS
# =============================================================================
# A school measuring on 20 days can have run 20 tests or 200, and the two are
# different deployments. The app's ceiling is four automatic tests a day (one per
# 4-hour window plus the daily launch test), so anything above that is a person
# pressing the button. Definitions used here:
#   · first-of-day  = the day's earliest test, the closest proxy available for the
#     launch test — the table carries no flag saying which test fired why
#   · burst         = a test within PING_BURST_MIN minutes of the same school's
#     previous one; the scheduler never does this, so it is a manual re-run
#   · the duplicate `measurement_uuid` rate is reported because it inflates any
#     per-day count and is invisible unless looked for
INTENSITY_CEILING = 4       # automatic tests a day the scheduler can produce
INTENSITY_BURST_MIN = 15    # minutes; closer than this is a manual re-run

Q_INTENSITY = f"""
WITH m AS (
    SELECT iso3_code, school_id_giga, measurement_uuid,
           CAST({_LTS} AS date) AS d, {_LTS} AS ts
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_scope_sql}
),
lagged AS (
    SELECT iso3_code, school_id_giga, d, ts,
           date_diff('minute', LAG(ts) OVER (PARTITION BY school_id_giga, d ORDER BY ts), ts) AS since_prev
    FROM m
),
per_day AS (
    SELECT iso3_code, school_id_giga, d, count(*) AS n_tests,
           count_if(since_prev IS NOT NULL AND since_prev < {INTENSITY_BURST_MIN}) AS n_burst
    FROM lagged GROUP BY 1, 2, 3
)
SELECT p.iso3_code,
       count(DISTINCT p.school_id_giga) AS schools,
       count(*) AS school_days,
       sum(p.n_tests) AS tests,
       approx_percentile(CAST(p.n_tests AS double), 0.5)  AS median_tests_per_day,
       approx_percentile(CAST(p.n_tests AS double), 0.9)  AS p90_tests_per_day,
       100.0 * avg(IF(p.n_tests = 1, 1e0, 0e0))           AS pct_days_single_test,
       100.0 * avg(IF(p.n_tests > {INTENSITY_CEILING}, 1e0, 0e0)) AS pct_days_above_ceiling,
       100.0 * sum(p.n_burst) / sum(p.n_tests)            AS pct_tests_in_burst,
       sum(p.n_tests) / CAST(count(DISTINCT p.school_id_giga) AS double) AS tests_per_school
FROM per_day p GROUP BY 1
HAVING count(DISTINCT p.school_id_giga) >= {MIN_SCHOOLS}
ORDER BY 5 DESC
"""

Q_INTENSITY_HIST = f"""
WITH per_day AS (
    SELECT iso3_code, school_id_giga, CAST({_LTS} AS date) AS d, count(*) AS n_tests
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_scope_sql}
    GROUP BY 1, 2, 3
)
SELECT iso3_code, LEAST(n_tests, 21) AS n_tests, count(*) AS n
FROM per_day GROUP BY 1, 2
"""

Q_DUP_UUID = f"""
SELECT count(*) AS rows_, count(DISTINCT measurement_uuid) AS uuids
FROM default.all_gigameter_measurement_data
WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_scope_sql}
"""

try:
    intensity = _pq(Q_INTENSITY).set_index('iso3_code')
    intensity_hist = _pq(Q_INTENSITY_HIST)
    _dup = _pq(Q_DUP_UUID).iloc[0]

    _show = ['schools', 'school_days', 'tests', 'median_tests_per_day', 'p90_tests_per_day',
             'pct_days_single_test', 'pct_days_above_ceiling', 'pct_tests_in_burst',
             'tests_per_school']
    display(intensity[_show].round(1).style
            .background_gradient(cmap='RdYlGn', subset=['median_tests_per_day'])
            .background_gradient(cmap='RdYlGn_r', subset=['pct_days_single_test'])
            .format('{:,.0f}', subset=['schools', 'school_days', 'tests'])
            .format('{:.1f}', subset=[c for c in _show
                                      if c not in ('schools', 'school_days', 'tests')]))

    _w = intensity['school_days']
    _f = {c: float((intensity[c] * _w).sum() / _w.sum())
          for c in ['median_tests_per_day', 'p90_tests_per_day', 'pct_days_single_test',
                    'pct_days_above_ceiling', 'pct_tests_in_burst']}
    _h = intensity_hist[intensity_hist['iso3_code'].isin(intensity.index)].groupby('n_tests')['n'].sum()
    print(f"FLEET: {int(intensity['tests'].sum()):,} tests over "
          f"{int(_w.sum()):,} school-days from {int(intensity['schools'].sum()):,} schools · "
          f"median {_f['median_tests_per_day']:.0f} tests a day, p90 {_f['p90_tests_per_day']:.0f}")
    print(f"        {_f['pct_days_single_test']:.0f}% of school-days carry a SINGLE test — one reading "
          f"is the whole day's evidence, and it is almost always a morning one (Part 3e)")
    print(f"        {_f['pct_days_above_ceiling']:.1f}% of school-days exceed the scheduler's ceiling of "
          f"{INTENSITY_CEILING}, so someone was pressing the button; "
          f"{_f['pct_tests_in_burst']:.1f}% of all tests land within {INTENSITY_BURST_MIN} min of the "
          f"previous one")
    print(f"        duplicate measurement_uuid: {100 * (1 - _dup['uuids'] / _dup['rows_']):.1f}% of rows "
          f"({int(_dup['rows_'] - _dup['uuids']):,} of {int(_dup['rows_']):,}) — they inflate every "
          f"per-day count above and are not de-duplicated upstream")
    print(f"        intensity vs presence: the median school runs "
          f"{intensity['tests_per_school'].median():.0f} tests in total, but the fleet spread is "
          f"{intensity['tests_per_school'].min():.0f} ({intensity['tests_per_school'].idxmin()}) to "
          f"{intensity['tests_per_school'].max():.0f} ({intensity['tests_per_school'].idxmax()})")

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.4), gridspec_kw={'width_ratios': [1, 1, 1.1]})
    axes[0].bar(_h.index, 100 * _h / _h.sum(), color=GIGA_PRIMARY[500], width=.85)
    axes[0].axvline(INTENSITY_CEILING + .5, color=GIGA_BAD, lw=1.4)
    axes[0].annotate(f'scheduler ceiling {INTENSITY_CEILING}',
                     xy=(INTENSITY_CEILING + .8, axes[0].get_ylim()[1] * .8), fontsize=8, color=GIGA_BAD)
    axes[0].set_xlabel('tests in a school-day (21 = 21 or more)')
    axes[0].set_ylabel('% of school-days'); axes[0].set_title('How many tests a measuring day carries')

    _p = intensity.sort_values('median_tests_per_day')
    _y = np.arange(len(_p))
    axes[1].barh(_y, _p['median_tests_per_day'], color=GIGA_PRIMARY[600], height=.55, label='median')
    axes[1].barh(_y, _p['p90_tests_per_day'], color=GIGA_PRIMARY[200], height=.55,
                 left=0, zorder=0, label='p90')
    axes[1].set_yticks(_y); axes[1].set_yticklabels(_p.index, fontsize=8)
    axes[1].set_xlabel('tests per school-day'); axes[1].legend(fontsize=8)
    axes[1].set_title('Median and p90 intensity')

    axes[2].scatter(intensity['pct_days_single_test'], intensity['pct_days_above_ceiling'],
                    s=np.sqrt(intensity['schools']) * 8, color=GIGA_PRIMARY[700],
                    alpha=.75, edgecolors='white', linewidths=.8)
    for _i, _r in intensity.iterrows():
        axes[2].annotate(_i, (_r['pct_days_single_test'], _r['pct_days_above_ceiling']),
                         fontsize=7.5, xytext=(4, 3), textcoords='offset points')
    axes[2].set_xlabel('% of school-days with a single test')
    axes[2].set_ylabel(f'% of school-days above the ceiling of {INTENSITY_CEILING}')
    axes[2].set_title('Thin days vs hand-driven days')
    plt.tight_layout(); plt.show()

    print("Reading it: a measuring day is a handful of readings clustered in the morning, not a day of")
    print("monitoring — and a fifth of them are a single reading. That caps what any daily statistic")
    print("can mean. It also says the days above the ceiling are a different population (someone is")
    print("demonstrating the app, and a quarter of all tests are re-runs minutes apart), which both")
    print("Part 3l and Part 3m have to hold separate from the scheduled fleet.")
except Exception as _e:
    intensity = pd.DataFrame(); intensity_hist = pd.DataFrame()
    print(f"(intensity needs Trino: {_e})")

---
## Part 3l — Concentration: who actually produces the data

Every country number in this notebook is an average over schools, and an average
describes the typical school only if schools contribute comparably. They do not.

Gini and the top-decile share say how top-heavy each fleet is, which sets a hard
limit on what a country-level statistic can mean: where a tenth of the schools
carry half the readings, the country's "average speed" is those schools' ISPs,
those schools' hours and those schools' equipment.

In [ ]:
# =============================================================================
# CONCENTRATION — how much of a country's data comes from how few schools
# =============================================================================
# Every country-level number in this notebook is an average over schools, and an
# average is only a description of the typical school if the schools contribute
# comparably. They do not. Gini is computed over measurements per school on the
# standard order-statistic form,
#     G = 2 * sum(i * x_i) / (n * sum(x_i)) - (n + 1) / n,   x ascending,
# so 0 is every school contributing equally and 1 is one school contributing
# everything. The top-decile share is the same fact in a form that can be quoted.
# Read alongside Part 3k: concentration here is in TESTS, and a country can be
# concentrated because a few schools measure often or because a few last longer.
Q_CONCENTRATION = f"""
WITH per_school AS (
    SELECT iso3_code, school_id_giga,
           count(*) AS tests,
           count(DISTINCT CAST({_LTS} AS date)) AS days
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_scope_sql}
    GROUP BY 1, 2
),
ranked AS (
    SELECT iso3_code, school_id_giga, tests, days,
           ROW_NUMBER() OVER (PARTITION BY iso3_code ORDER BY tests) AS i_tests,
           ROW_NUMBER() OVER (PARTITION BY iso3_code ORDER BY days)  AS i_days,
           count(*)  OVER (PARTITION BY iso3_code) AS n,
           sum(tests) OVER (PARTITION BY iso3_code) AS tot_tests,
           sum(days)  OVER (PARTITION BY iso3_code) AS tot_days
    FROM per_school
)
SELECT iso3_code,
       max(n) AS schools,
       max(tot_tests) AS tests,
       2.0 * sum(i_tests * tests) / (max(n) * max(tot_tests)) - (max(n) + 1.0) / max(n) AS gini_tests,
       2.0 * sum(i_days  * days)  / (max(n) * max(tot_days))  - (max(n) + 1.0) / max(n) AS gini_days,
       100.0 * sum(IF(i_tests > 0.9 * n, tests, 0)) / max(tot_tests)  AS pct_tests_top10,
       100.0 * sum(IF(i_tests > 0.99 * n, tests, 0)) / max(tot_tests) AS pct_tests_top1,
       100.0 * sum(IF(i_tests <= 0.5 * n, tests, 0)) / max(tot_tests) AS pct_tests_bottom50,
       approx_percentile(CAST(tests AS double), 0.5) AS median_tests,
       max(tot_tests) / CAST(max(n) AS double) AS mean_tests
FROM ranked GROUP BY 1
HAVING max(n) >= {MIN_SCHOOLS}
ORDER BY 4 DESC
"""

# The Lorenz curve itself, in 20 buckets — enough to draw, small enough to return.
Q_LORENZ = f"""
WITH per_school AS (
    SELECT iso3_code, school_id_giga, count(*) AS tests
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL {_TZ_DROP_SQL} {_scope_sql}
    GROUP BY 1, 2
),
ranked AS (
    SELECT iso3_code, tests,
           NTILE(20) OVER (PARTITION BY iso3_code ORDER BY tests) AS bucket
    FROM per_school
)
SELECT iso3_code, bucket, sum(tests) AS tests, count(*) AS schools
FROM ranked GROUP BY 1, 2
"""

try:
    conc = _pq(Q_CONCENTRATION).set_index('iso3_code')
    lorenz = _pq(Q_LORENZ)
    lorenz = lorenz[lorenz['iso3_code'].isin(conc.index)]

    _show = ['schools', 'tests', 'gini_tests', 'gini_days', 'pct_tests_top10', 'pct_tests_top1',
             'pct_tests_bottom50', 'median_tests', 'mean_tests']
    display(conc[_show].round(2).style
            .background_gradient(cmap='RdYlGn_r', subset=['gini_tests', 'pct_tests_top10'])
            .format('{:,.0f}', subset=['schools', 'tests', 'median_tests', 'mean_tests'])
            .format('{:.2f}', subset=['gini_tests', 'gini_days'])
            .format('{:.1f}', subset=['pct_tests_top10', 'pct_tests_top1', 'pct_tests_bottom50']))

    _w = conc['tests']
    _g = float((conc['gini_tests'] * _w).sum() / _w.sum())
    _t10 = float((conc['pct_tests_top10'] * _w).sum() / _w.sum())
    _b50 = float((conc['pct_tests_bottom50'] * _w).sum() / _w.sum())
    print(f"FLEET: Gini {_g:.2f} on tests per school · the busiest 10% of schools produce "
          f"{_t10:.0f}% of all measurements, the quieter half produces {_b50:.0f}%")
    _ratio = (conc['mean_tests'] / conc['median_tests'])
    print(f"        the mean school is {_ratio.median():.1f}x the median school in the typical country "
          f"(worst {_ratio.max():.1f}x in {_ratio.idxmax()}, best {_ratio.min():.1f}x in "
          f"{_ratio.idxmin()}) — a country MEAN is a statement about the busy tail, which is why the "
          f"medians in Part 4 are the ones to quote")
    print(f"        most concentrated {conc['gini_tests'].idxmax()} "
          f"(Gini {conc['gini_tests'].max():.2f}, top decile "
          f"{conc.loc[conc['gini_tests'].idxmax(), 'pct_tests_top10']:.0f}%) · "
          f"most even {conc['gini_tests'].idxmin()} (Gini {conc['gini_tests'].min():.2f})")
    print(f"        Gini is lower on DAYS measured ({float((conc['gini_days'] * _w).sum() / _w.sum()):.2f}) "
          f"than on tests, so schools differ more in how hard they measure than in how long they last")

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.4), gridspec_kw={'width_ratios': [1, 1, 1.1]})

    _piv = (lorenz.pivot_table(index='iso3_code', columns='bucket', values='tests', fill_value=0)
                  .reindex(conc.sort_values('gini_tests').index))
    _cum = _piv.div(_piv.sum(axis=1), axis=0).cumsum(axis=1) * 100
    _x = np.arange(0, 105, 5)
    axes[0].plot(_x, _x, color=GIGA_GREY[400], ls='--', lw=1, label='perfect equality')
    for _i, _iso in enumerate(_cum.index):
        _c = GIGA_PRIMARY[300] if _i < len(_cum) - 3 else GIGA_BAD
        axes[0].plot(_x, [0] + list(_cum.loc[_iso]), color=_c, lw=1.1, alpha=.75)
    axes[0].annotate(f'{_cum.index[-1]} (most concentrated)', xy=(60, 8), fontsize=8, color=GIGA_BAD)
    axes[0].set_xlabel('% of schools, least busy first'); axes[0].set_ylabel('% of measurements')
    axes[0].set_title('Lorenz curves'); axes[0].legend(fontsize=8, loc='upper left')

    _p = conc.sort_values('pct_tests_top10')
    _y = np.arange(len(_p))
    axes[1].barh(_y, _p['pct_tests_top10'], color=GIGA_PRIMARY[600], height=.6)
    axes[1].axvline(10, color=GIGA_GREY[400], ls='--', lw=1)
    axes[1].annotate('even fleet = 10%', xy=(11, .3), fontsize=8, color=GIGA_GREY[500])
    axes[1].set_yticks(_y); axes[1].set_yticklabels(_p.index, fontsize=8)
    axes[1].set_xlabel('% of measurements from the busiest 10% of schools')
    axes[1].set_title('How top-heavy each fleet is')

    axes[2].scatter(conc['median_tests'], conc['mean_tests'],
                    s=np.sqrt(conc['schools']) * 8, color=GIGA_PRIMARY[700],
                    alpha=.75, edgecolors='white', linewidths=.8)
    _lim = max(conc['mean_tests'].max(), conc['median_tests'].max()) * 1.05
    axes[2].plot([0, _lim], [0, _lim], color=GIGA_GREY[400], ls='--', lw=1)
    for _i, _r in conc.iterrows():
        axes[2].annotate(_i, (_r['median_tests'], _r['mean_tests']), fontsize=7.5,
                         xytext=(4, 3), textcoords='offset points')
    axes[2].set_xlabel('median tests per school'); axes[2].set_ylabel('mean tests per school')
    axes[2].set_title('Every country sits above the line')
    plt.tight_layout(); plt.show()

    print("Reading it: the gap between the mean and the median school is the size of the reporting")
    print("error you make by quoting a fleet average. It also sets the ceiling on representativeness:")
    print("a country whose top decile carries most of the data is describing those schools' ISPs and")
    print("those schools' hours, whatever the school count in the header says.")
except Exception as _e:
    conc = pd.DataFrame(); lorenz = pd.DataFrame()
    print(f"(concentration needs Trino: {_e})")

---
## Part 3m — How much measurement is enough?

Two questions the row counts cannot answer.

**Where does the variation live** — between schools, between days at one school,
or between tests within a single day? The answer decides whether the scheduler's
three windows exist to buy *precision* or merely *coverage*.

**How many readings does a school's median need** before it stops moving? An
n-sample median is scored against the same school's other readings, which is the
empirical version of `MIN_MEASUREMENTS_FOR_IQB` — currently a chosen threshold
rather than a derived one.

In [ ]:
# =============================================================================
# HOW MUCH MEASUREMENT IS ENOUGH — variance structure, then a stability curve
# =============================================================================
# Two questions the fleet cannot answer by counting rows. FIRST, where does the
# variation in a school's download speed live: between schools, between days at
# one school, or between tests within one day? If within-day variance is small,
# repeat tests in a day buy little and the scheduler's three windows are about
# COVERAGE, not precision. SECOND, how many measurements does a school's median
# need before it stops moving — the empirical version of MIN_MEASUREMENTS_FOR_IQB.
#   · speeds are logged (log10) before any variance is taken; download is
#     right-skewed and a raw variance would describe the fastest schools only
#   · only rows that pass the DQ flag are used — `pass_fail_overall` is a
#     data-validity flag, not a verdict on the connection
#   · the decomposition uses one coherent cohort: days with >= 3 tests, schools
#     with >= 10 such days, so all three terms describe the same schools
#   · the variance terms are unweighted means of within-group variances, which is
#     the standard approximation, not an exact nested ANOVA
VAR_MIN_TESTS_DAY  = 3       # tests in a day for the within-day term to exist
VAR_MIN_DAYS       = 10      # such days a school needs to enter the decomposition
BOOT_MIN_TESTS     = 50      # a school needs this many tests to be a bootstrap target
BOOT_SCHOOLS_CTY   = 40      # schools sampled per country
BOOT_CAP           = 400     # measurements taken per school
BOOT_N             = [1, 2, 3, 5, 10, 20, 30, 50, 100]
BOOT_REPS          = 200
BOOT_HOLDOUT       = 30      # the estimate of n is scored against this many OTHER readings of the
                             # same school, never against a set containing them — resampling with
                             # replacement and scoring against the same values drives the error to
                             # zero as n grows, which would make any threshold look reachable

_M_OK = f"""
    SELECT iso3_code, school_id_giga, CAST({_LTS} AS date) AS d,
           log10(CAST(download_speed AS double)) AS v
    FROM default.all_gigameter_measurement_data
    WHERE local_created_timestamp IS NOT NULL AND download_speed > 0
      AND pass_fail_overall = 'pass' {_TZ_DROP_SQL} {_scope_sql}
"""

Q_VARIANCE = f"""
WITH m AS ({_M_OK}),
sd AS (
    SELECT iso3_code, school_id_giga, d, avg(v) AS m_day, var_pop(v) AS v_within, count(*) AS n
    FROM m GROUP BY 1, 2, 3 HAVING count(*) >= {VAR_MIN_TESTS_DAY}
),
sc AS (
    SELECT iso3_code, school_id_giga, avg(m_day) AS m_school, var_pop(m_day) AS v_day,
           avg(v_within) AS v_within, count(*) AS days
    FROM sd GROUP BY 1, 2 HAVING count(*) >= {VAR_MIN_DAYS}
)
SELECT iso3_code, count(*) AS schools, sum(days) AS school_days,
       var_pop(m_school) AS var_between_schools,
       avg(v_day)        AS var_between_days,
       avg(v_within)     AS var_within_day
FROM sc GROUP BY 1 HAVING count(*) >= {MIN_SCHOOLS}
"""

Q_BOOT_SAMPLE = f"""
WITH m AS ({_M_OK}),
elig AS (
    SELECT iso3_code, school_id_giga, count(*) AS n FROM m
    GROUP BY 1, 2 HAVING count(*) >= {BOOT_MIN_TESTS}
),
pick AS (
    SELECT iso3_code, school_id_giga,
           ROW_NUMBER() OVER (PARTITION BY iso3_code ORDER BY school_id_giga) AS rn
    FROM elig
),
rows_ AS (
    SELECT m.iso3_code, m.school_id_giga, m.v,
           ROW_NUMBER() OVER (PARTITION BY m.school_id_giga ORDER BY m.d) AS k
    FROM m JOIN pick p
      ON p.iso3_code = m.iso3_code AND p.school_id_giga = m.school_id_giga
    WHERE p.rn <= {BOOT_SCHOOLS_CTY}
)
SELECT iso3_code, school_id_giga, v FROM rows_ WHERE k <= {BOOT_CAP}
"""

try:
    var_c = _pq(Q_VARIANCE).set_index('iso3_code')
    boot_raw = _pq(Q_BOOT_SAMPLE)

    _tot = var_c[['var_between_schools', 'var_between_days', 'var_within_day']].sum(axis=1)
    _share = 100 * var_c[['var_between_schools', 'var_between_days', 'var_within_day']].div(_tot, axis=0)
    _share.columns = ['% between schools', '% between days', '% within a day']
    _out = var_c[['schools', 'school_days']].join(_share).sort_values('% within a day')
    display(_out.round(1).style
            .background_gradient(cmap='Blues', subset=list(_share.columns))
            .format('{:,.0f}', subset=['schools', 'school_days'])
            .format('{:.1f}', subset=list(_share.columns)))

    _w = var_c['schools']
    _fs = {c: float((_share[c] * _w).sum() / _w.sum()) for c in _share.columns}
    print(f"Variance in log10 download, cohort = {int(var_c['schools'].sum()):,} schools with "
          f">={VAR_MIN_DAYS} days of >={VAR_MIN_TESTS_DAY} tests:")
    print(f"        between schools {_fs['% between schools']:.0f}% · "
          f"between days at one school {_fs['% between days']:.0f}% · "
          f"within a single day {_fs['% within a day']:.0f}%")
    _sd_within = float(np.sqrt((var_c['var_within_day'] * _w).sum() / _w.sum()))
    _sd_day = float(np.sqrt((var_c['var_between_days'] * _w).sum() / _w.sum()))
    print(f"        two readings from the SAME day sit a factor of x{10 ** _sd_within:.2f} apart "
          f"(sd {_sd_within:.3f} in log10) — barely tighter than two readings from DIFFERENT days, "
          f"x{10 ** _sd_day:.2f}. A single reading is therefore a weak estimate of the day, and repeat "
          f"tests inside a day are NOT redundant: the within-day term is the largest of the three.")
    print(f"        caveat: most tests cluster in the morning (Part 3e), so this within-day spread is "
          f"mostly close-in-time variability, not the morning-to-afternoon difference — the schedule "
          f"never samples the day widely enough to separate the two.")

    # ---- stability curve: an n-sample median scored against the school's OTHER readings
    _g = boot_raw.groupby(['iso3_code', 'school_id_giga'])['v'].apply(lambda s: s.to_numpy(dtype=float))
    _rng = np.random.default_rng(20260814)
    _rows = []
    for (_iso, _sid), _v in _g.items():
        _N = len(_v)
        if _N < BOOT_MIN_TESTS:
            continue
        for _n in BOOT_N:
            if _N < _n + BOOT_HOLDOUT:      # not enough held-out readings to score against
                continue
            _err = np.empty(BOOT_REPS)
            for _b in range(BOOT_REPS):
                _perm = _rng.permutation(_N)
                _est = np.median(_v[_perm[:_n]])
                _truth = np.median(_v[_perm[_n:]])
                _err[_b] = 100 * abs(10 ** (_est - _truth) - 1)
            _rows.append({'iso3_code': _iso, 'school_id_giga': _sid, 'n': _n,
                          'median_err': float(np.median(_err)),
                          'p90_err': float(np.percentile(_err, 90))})
    boot = pd.DataFrame(_rows)

    _curve = boot.groupby('n')[['median_err', 'p90_err']].median()
    _sup = boot.groupby('n')['school_id_giga'].nunique()
    print(f"\nStability of a school's median download — n readings scored against >={BOOT_HOLDOUT} OTHER "
          f"readings of the same school, {boot['school_id_giga'].nunique():,} schools, "
          f"{BOOT_REPS} splits each:")
    print("     n   typical error   p90 error   schools")
    for _n in BOOT_N:
        if _n in _curve.index:
            print(f"  {_n:>4}       {_curve.loc[_n, 'median_err']:5.1f}%       "
                  f"{_curve.loc[_n, 'p90_err']:5.1f}%     {_sup[_n]:,}")
    _hit = [_n for _n in _curve.index if _curve.loc[_n, 'median_err'] < 10]
    print(f"        typical error falls below 10% at n = "
          f"{_hit[0] if _hit else 'more than ' + str(int(_curve.index.max()))} · "
          f"MIN_MEASUREMENTS_FOR_IQB is currently {MIN_MEASUREMENTS_FOR_IQB}, which sits at a typical "
          f"{np.interp(MIN_MEASUREMENTS_FOR_IQB, _curve.index, _curve['median_err']):.0f}% error and a "
          f"p90 of {np.interp(MIN_MEASUREMENTS_FOR_IQB, _curve.index, _curve['p90_err']):.0f}%")
    print(f"        the curve flattens rather than converging: the floor is the school's own "
          f"day-to-day and within-day spread, so no amount of sampling makes a single number describe "
          f"a school that genuinely varies.")

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.4), gridspec_kw={'width_ratios': [1.1, 1, 1]})

    _p = _out[list(_share.columns)].iloc[::-1]
    _y = np.arange(len(_p)); _left = np.zeros(len(_p))
    for _c, _col in zip(_share.columns, [GIGA_PRIMARY[800], GIGA_PRIMARY[500], GIGA_PRIMARY[200]]):
        axes[0].barh(_y, _p[_c], left=_left, color=_col, label=_c, height=.75)
        _left = _left + _p[_c].values
    axes[0].set_yticks(_y); axes[0].set_yticklabels(_p.index, fontsize=8)
    axes[0].set_xlim(0, 100); axes[0].set_xlabel('% of variance in log10 download')
    axes[0].set_title('Where the variation lives'); axes[0].legend(fontsize=7, loc='lower right')

    _ns = list(_curve.index)
    _cty = boot.groupby(['iso3_code', 'n'])['median_err'].median().unstack()
    for _iso in _cty.index:
        axes[1].plot(_cty.columns, _cty.loc[_iso], color=GIGA_PRIMARY[300], lw=1, alpha=.7)
    axes[1].plot(_ns, _curve['p90_err'], color=GIGA_BAD, lw=2, label='fleet p90')
    axes[1].plot(_ns, _curve['median_err'], color=GIGA_PRIMARY[800], lw=2, ls='--',
                 label='fleet typical')
    axes[1].axhline(10, color=GIGA_GREY[400], ls=':', lw=1)
    axes[1].axvline(MIN_MEASUREMENTS_FOR_IQB, color=GIGA_BLUE, lw=1.2)
    axes[1].annotate(f'IQB min = {MIN_MEASUREMENTS_FOR_IQB}',
                     xy=(MIN_MEASUREMENTS_FOR_IQB * 1.1, axes[1].get_ylim()[1] * .8),
                     fontsize=8, color=GIGA_BLUE)
    axes[1].set_xscale('log'); axes[1].set_xticks(_ns)
    axes[1].set_xticklabels([str(_n) for _n in _ns], fontsize=8)
    axes[1].set_xlabel('measurements used'); axes[1].set_ylabel('% error on the school median')
    axes[1].set_title('How fast a school median settles'); axes[1].legend(fontsize=8)

    # what a day actually delivers, against what it would need
    _dist = (intensity_hist[intensity_hist['iso3_code'].isin(var_c.index)]
             .groupby('n_tests')['n'].sum() if len(intensity_hist) else pd.Series(dtype=float))
    if len(_dist):
        _cumd = 100 * _dist.sort_index().cumsum() / _dist.sum()
        axes[2].plot(_cumd.index, 100 - _cumd.values, color=GIGA_PRIMARY[700], lw=2)
        _need = _hit[0] if _hit else int(_curve.index.max())
        axes[2].axvline(_need, color=GIGA_BAD, lw=1.4)
        axes[2].annotate(f'n for a typical error < 10%', xy=(_need * 1.1, 60), fontsize=8, color=GIGA_BAD)
        axes[2].set_xscale('log')
        axes[2].set_xlabel('tests in a school-day'); axes[2].set_ylabel('% of school-days with at least this many')
        axes[2].set_title('What a single day can deliver')
    plt.tight_layout(); plt.show()

    print("Reading it: the variation splits roughly in thirds — between schools, between days, and")
    print("WITHIN a single day. That last third is the uncomfortable one, because it means one reading")
    print("does not describe a day and Part 3k found that a fifth of school-days carry exactly one.")
    print("Both levers are therefore real and they are not interchangeable: more tests inside a day cut")
    print("the within-day third, more DAYS cut the between-day third, and neither touches the floor set")
    print("by how much a school genuinely varies. Against this curve the IQB minimum is a tolerance")
    print("choice rather than a convergence point, and it should be stated as one.")
except Exception as _e:
    var_c = pd.DataFrame(); boot = pd.DataFrame(); boot_raw = pd.DataFrame()
    print(f"(variance and stability need Trino: {_e})")

---
## Part 4 — Fleet comparison

One row per country: rhythm, seasonality and churn side by side. This is the table
to track over time and to hand to country teams.

In [ ]:
# =============================================================================
# FLEET PROFILE — one row per country
# =============================================================================
fleet = pd.DataFrame({
    'schools': profile['schools'].astype(int),
    'data span (mo)': ((pd.to_datetime(profile['data_to']) - pd.to_datetime(profile['data_from']))
                       .dt.days / 30.44).round(0),
    'median tenure (mo)': profile['median_tenure_months'].round(1),
    'median days/active mo': profile['median_days_per_month'].round(1),
    'gap = 1 day %': rhythm['pct_gap_1d'].round(0),
    'weekday %': rhythm['pct_weekday'].round(0),
    # index-aligned Series everywhere — a positional list here silently mismatches
    # rows when the frames carry differently ordered indexes
    'low season': pd.Series({i: ', '.join(_MON[c - 1] for c in LOW_SEASON.get(i, [])) or '—'
                             for i in profile.index}),
    'trough % of median': season_summary['trough % of median'].reindex(profile.index),
    'silent >90d now %': profile['pct_silent_90d'].where(profile['n_eligible_90d'] >= 20).round(0),
    'n eligible 90d': profile['n_eligible_90d'].astype(int),
    'P(return | 90d) % (spells)': churn['P(return | 90d) %'],
    'churn /100 yrs @90d': churn['churn/100 yrs @90d'],
    'churn /100 yrs @182d': churn['churn/100 yrs @182d'],
    'campaign in window': pd.Series({i: ('yes: ' + '; '.join(l for _, _, l in campaign_windows(i)))
                                     if CAMPAIGNS.get(i) else 'no' for i in profile.index}),
})
fleet = fleet.reindex(profile.index)      # keep one canonical row order
_rhythm_label = np.select(
    [fleet['median days/active mo'] >= 15, fleet['median days/active mo'] >= 8],
    ['near-daily', 'several days/week'], default='sparse')
fleet.insert(4, 'rhythm', _rhythm_label)
display(fleet)

_out = CACHE_ROOT / '_fleet'
_out.mkdir(exist_ok=True)
fleet.to_csv(_out / 'fleet_profile.csv')
season.to_csv(_out / 'fleet_seasonality.csv', index=False)
pd.concat({k: v for k, v in surv_tables.items()}, names=['iso3_code']).to_csv(_out / 'fleet_survival.csv')
print(f"exported to {_out}/ (fleet_profile, fleet_seasonality, fleet_survival)")
print("\nReading guide: 'rhythm' and 'low season' describe how a fleet behaves; "
      "'P(return | 90d)' and\n'churn /100 school-yrs' describe how much of it is being lost — "
      "the second pair is only\ncomparable across countries because both are exposure-corrected.")